
# 13 — Final Project Technical Validation

## Objective
Independently validate the historical demand, forecasting, external signals, inventory simulations, sensitivity analysis, and reporting warehouse used in the Hardware Demand-Sensing & Launch Readiness Planner.

## Validation Principles
- Read existing outputs without modifying source data.
- Recalculate important KPIs independently.
- Reconcile analytical results with dashboard reporting data.
- Document any discrepancies and limitations.
- Confirm the project can be reproduced.

## Expected Outcome
A documented PASS/FAIL audit supporting the final GitHub portfolio case study.


In [1]:
# STEP 1: INITIALIZE FINAL PROJECT VALIDATION

from pathlib import Path
import pandas as pd
import duckdb

# Locate project root
PROJECT_ROOT = Path.cwd().resolve()

while (
    PROJECT_ROOT.parent != PROJECT_ROOT
    and not (PROJECT_ROOT / "data").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "data").is_dir(), (
    "Project data folder not found."
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WAREHOUSE_DIR = PROJECT_ROOT / "data" / "warehouse"
DASHBOARD_DIR = PROJECT_ROOT / "data" / "dashboard"
CONFIG_DIR = PROJECT_ROOT / "config"

print("===== PROJECT CONFIGURATION =====")
print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Warehouse:", WAREHOUSE_DIR)
print("Dashboard exports:", DASHBOARD_DIR)
print("Configuration:", CONFIG_DIR)

# Inventory existing project outputs
processed_files = sorted(PROCESSED_DIR.glob("*.csv"))
warehouse_files = sorted(WAREHOUSE_DIR.glob("*.duckdb"))
dashboard_files = sorted(DASHBOARD_DIR.rglob("*.csv"))

print("\n===== PROJECT FILE INVENTORY =====")
print("Processed CSV files:", len(processed_files))
print("DuckDB warehouse files:", len(warehouse_files))
print("Dashboard CSV exports:", len(dashboard_files))

print("\n===== AVAILABLE WAREHOUSES =====")
for file in warehouse_files:
    print(file.name)

print("\n===== CRITICAL FILE CHECKS =====")

critical_files = {
    "AirPods historical demand":
        PROCESSED_DIR / "airpods_weekly_2019.csv",
    "External signal model comparison":
        PROCESSED_DIR / "external_signal_model_comparison_2019.csv",
    "External signal stability":
        PROCESSED_DIR / "external_signal_stability_2019.csv",
}

for label, path in critical_files.items():
    status = "PASS" if path.exists() else "MISSING"
    print(f"{label}: {status}")

print("\n===== STEP 1 COMPLETE =====")
print("Read-only setup: no source files modified.")


===== PROJECT CONFIGURATION =====
Project root: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner
Processed data: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/processed
Warehouse: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse
Dashboard exports: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/dashboard
Configuration: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/config

===== PROJECT FILE INVENTORY =====
Processed CSV files: 43
DuckDB warehouse files: 5
Dashboard CSV exports: 28

===== AVAILABLE WAREHOUSES =====
demand_intelligence.duckdb
demand_intelligence_manifest_only.duckdb
demand_intelligence_rebuild_candidate.duckdb
demand_intelligence_signals_candidate.duckdb
demand_intelligence_signals_manifest_rebuild.duckdb

===== CRITICAL FILE CHECKS =====
AirPods historical demand: PASS
Externa

In [2]:
# STEP 2: HISTORICAL AIRPODS DEMAND RECONCILIATION

import pandas as pd
import duckdb

print("===== HISTORICAL DEMAND VALIDATION =====")

# 1. Load original processed historical dataset
historical_path = PROCESSED_DIR / "airpods_weekly_2019.csv"

historical_df = pd.read_csv(historical_path)

print("\n===== SOURCE DATASET =====")
print("File:", historical_path.name)
print("Rows:", len(historical_df))
print("Columns:", historical_df.columns.tolist())

# 2. Standardize the two required fields
required_columns = ["Week_Ending", "Units_Sold"]

assert all(
    col in historical_df.columns for col in required_columns
), "Required historical demand columns missing."

def prepare_weekly_data(df):
    result = df[["Week_Ending", "Units_Sold"]].copy()

    result["Week_Ending"] = pd.to_datetime(
        result["Week_Ending"], errors="raise"
    )

    result["Units_Sold"] = pd.to_numeric(
        result["Units_Sold"], errors="raise"
    )

    return result.sort_values(
        "Week_Ending"
    ).reset_index(drop=True)

historical_clean = prepare_weekly_data(historical_df)

# 3. Independently calculate historical KPIs
historical_weeks = len(historical_clean)
historical_units = historical_clean["Units_Sold"].sum()
average_weekly_units = historical_clean["Units_Sold"].mean()

print("\n===== RECALCULATED HISTORICAL KPIS =====")
print("Historical weeks:", historical_weeks)
print("Total units sold:", f"{historical_units:,.0f}")
print("Average weekly units:", f"{average_weekly_units:,.2f}")
print("Minimum weekly units:", historical_clean["Units_Sold"].min())
print("Maximum weekly units:", historical_clean["Units_Sold"].max())

# 4. Data integrity checks
assert historical_weeks == 51, "Historical week count mismatch."
assert historical_units == 15364, "Historical unit total mismatch."

assert historical_clean["Week_Ending"].notna().all()
assert historical_clean["Units_Sold"].notna().all()
assert historical_clean["Week_Ending"].is_unique
assert (historical_clean["Units_Sold"] >= 0).all()

print("\n===== HISTORICAL INTEGRITY =====")
print("51 historical weeks: PASS")
print("15,364 total units: PASS")
print("No missing required values: PASS")
print("No duplicate week endings: PASS")
print("Nonnegative demand values: PASS")

# 5. Reconcile Tableau dashboard export
dashboard_path = (
    DASHBOARD_DIR
    / "executive_v1"
    / "dashboard_v_airpods_weekly_demand.csv"
)

assert dashboard_path.exists(), (
    f"Dashboard export not found: {dashboard_path}"
)

dashboard_df = prepare_weekly_data(
    pd.read_csv(dashboard_path)
)

pd.testing.assert_frame_equal(
    historical_clean,
    dashboard_df,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-9
)

print("\n===== TABLEAU EXPORT RECONCILIATION =====")
print("Dashboard rows:", len(dashboard_df))
print("Dashboard units:", f"{dashboard_df['Units_Sold'].sum():,.0f}")
print("Week-by-week reconciliation: PASS")

# 6. Reconcile DuckDB reporting warehouse
warehouse_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert warehouse_path.exists(), "Final warehouse not found."

con = duckdb.connect(
    str(warehouse_path),
    read_only=True
)

try:
    warehouse_df = con.execute("""
        SELECT
            Week_Ending,
            Units_Sold
        FROM reporting.v_airpods_weekly_demand
        ORDER BY Week_Ending
    """).df()

finally:
    con.close()

warehouse_clean = prepare_weekly_data(warehouse_df)

pd.testing.assert_frame_equal(
    historical_clean,
    warehouse_clean,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-9
)

print("\n===== DUCKDB RECONCILIATION =====")
print("Warehouse rows:", len(warehouse_clean))
print("Warehouse units:", f"{warehouse_clean['Units_Sold'].sum():,.0f}")
print("Week-by-week reconciliation: PASS")

# 7. Final audit summary
validation_report = pd.DataFrame([
    {
        "Source": "Processed Historical CSV",
        "Weeks": len(historical_clean),
        "Units": historical_clean["Units_Sold"].sum(),
        "Status": "PASS"
    },
    {
        "Source": "Tableau Dashboard Export",
        "Weeks": len(dashboard_df),
        "Units": dashboard_df["Units_Sold"].sum(),
        "Status": "PASS"
    },
    {
        "Source": "DuckDB Reporting Warehouse",
        "Weeks": len(warehouse_clean),
        "Units": warehouse_clean["Units_Sold"].sum(),
        "Status": "PASS"
    }
])

print("\n===== HISTORICAL RECONCILIATION REPORT =====")
display(validation_report)

print("\n===== STEP 2 COMPLETE =====")
print("Historical demand reconciliation: PASS")
print("All sources verified without modifying data.")


===== HISTORICAL DEMAND VALIDATION =====

===== SOURCE DATASET =====
File: airpods_weekly_2019.csv
Rows: 51
Columns: ['Week_Ending', 'Units_Sold']

===== RECALCULATED HISTORICAL KPIS =====
Historical weeks: 51
Total units sold: 15,364
Average weekly units: 301.25
Minimum weekly units: 175
Maximum weekly units: 515

===== HISTORICAL INTEGRITY =====
51 historical weeks: PASS
15,364 total units: PASS
No missing required values: PASS
No duplicate week endings: PASS
Nonnegative demand values: PASS

===== TABLEAU EXPORT RECONCILIATION =====
Dashboard rows: 51
Dashboard units: 15,364
Week-by-week reconciliation: PASS

===== DUCKDB RECONCILIATION =====
Warehouse rows: 51
Warehouse units: 15,364
Week-by-week reconciliation: PASS

===== HISTORICAL RECONCILIATION REPORT =====


,Source,Weeks,Units,Status
0,Processed Historical CSV,51,15364.0,PASS
1,Tableau Dashboard Export,51,15364.0,PASS
2,DuckDB Reporting Warehouse,51,15364.0,PASS



===== STEP 2 COMPLETE =====
Historical demand reconciliation: PASS
All sources verified without modifying data.


In [4]:
# STEP 3: FORECAST MODEL PERFORMANCE RECONCILIATION

import pandas as pd
import duckdb

print("===== FORECAST VALIDATION CONFIGURATION =====")

warehouse_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

dashboard_path = (
    DASHBOARD_DIR
    / "executive_v1"
    / "dashboard_v_forecast_model_performance.csv"
)

assert warehouse_path.exists(), "Warehouse database missing."
assert dashboard_path.exists(), "Forecast dashboard CSV missing."

print("Warehouse:", warehouse_path.name)
print("Dashboard source:", dashboard_path.name)

# 1. Load forecast performance from both sources
dashboard_forecast = pd.read_csv(dashboard_path)

con = duckdb.connect(
    str(warehouse_path),
    read_only=True
)

try:
    warehouse_forecast = con.execute("""
        SELECT *
        FROM reporting.v_forecast_model_performance
    """).df()
finally:
    con.close()

print("\n===== FORECAST DATA INVENTORY =====")
print("Dashboard rows:", len(dashboard_forecast))
print("Warehouse rows:", len(warehouse_forecast))
print("Available columns:", dashboard_forecast.columns.tolist())

# 2. Verify full reporting dataset reconciliation
assert set(dashboard_forecast.columns) == set(
    warehouse_forecast.columns
), "Forecast columns differ."

columns = sorted(dashboard_forecast.columns)

def normalize_report(df):
    result = df[columns].copy()

    for col in result.columns:
        if pd.api.types.is_datetime64_any_dtype(result[col]):
            result[col] = result[col].astype(str)

    return result.sort_values(
        columns
    ).reset_index(drop=True)

pd.testing.assert_frame_equal(
    normalize_report(dashboard_forecast),
    normalize_report(warehouse_forecast),
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-6
)

print("\n===== REPORTING SOURCE RECONCILIATION =====")
print("All rows and fields match: PASS")

# 3. Filter the original AirPods forecasting experiment
required = ["Experiment", "Model", "WAPE_Pct"]

assert all(
    col in dashboard_forecast.columns for col in required
), "Required model comparison fields missing."

airpods_models = dashboard_forecast[
    dashboard_forecast["Experiment"]
    .astype(str)
    .str.contains("AirPods", case=False, na=False)
].copy()

airpods_models["WAPE_Pct"] = pd.to_numeric(
    airpods_models["WAPE_Pct"],
    errors="raise"
)

airpods_models = airpods_models.sort_values(
    "WAPE_Pct"
).reset_index(drop=True)

print("\n===== ORIGINAL AIRPODS FORECAST MODELS =====")

show_columns = [
    col for col in
    ["Experiment", "Model", "MAE", "WAPE_Pct", "Bias"]
    if col in airpods_models.columns
]

display(airpods_models[show_columns].round(2))

# 4. Validate benchmark results
assert len(airpods_models) == 5, (
    f"Expected 5 models, found {len(airpods_models)}."
)

best_model = airpods_models.iloc[0]

assert "naive" in str(best_model["Model"]).lower(), (
    "Naive model is not the lowest-WAPE model."
)

assert abs(float(best_model["WAPE_Pct"]) - 8.60) <= 0.01, (
    "Naive forecast WAPE does not match 8.60%."
)

assert airpods_models["WAPE_Pct"].notna().all()
assert (airpods_models["WAPE_Pct"] >= 0).all()

print("\n===== FORECAST BENCHMARK VALIDATION =====")
print("Original models compared: 5 - PASS")
print("Best model:", best_model["Model"])
print("Best WAPE:", f"{best_model['WAPE_Pct']:.2f}%")
print("Naive 8.60% benchmark: PASS")
print("Nonnegative, nonmissing WAPE values: PASS")

# 5. Find prediction-level evidence for independent validation
print("\n===== AVAILABLE PREDICTION / BACKTEST FILES =====")

prediction_files = sorted(
    path for path in PROCESSED_DIR.glob("*.csv")
    if any(
        word in path.name.lower()
        for word in ["forecast", "prediction", "backtest", "model"]
    )
)

for path in prediction_files:
    print(path.name)

print("\n===== STEP 3 COMPLETE =====")
print("Stored forecast performance reconciliation: PASS")
print("Next: Independent forecast-error recalculation.")


===== FORECAST VALIDATION CONFIGURATION =====
Warehouse: demand_intelligence_signals_manifest_rebuild.duckdb
Dashboard source: dashboard_v_forecast_model_performance.csv

===== FORECAST DATA INVENTORY =====
Dashboard rows: 8
Warehouse rows: 8
Available columns: ['Experiment', 'Target_Unit', 'Model', 'MAE', 'WAPE_Pct', 'Bias']

===== REPORTING SOURCE RECONCILIATION =====
All rows and fields match: PASS

===== ORIGINAL AIRPODS FORECAST MODELS =====


,Experiment,Model,MAE,WAPE_Pct,Bias
0,AirPods Weekly Units,Naive Forecast,22.16,8.60,-3.42
1,AirPods Weekly Units,4-Week Moving Average,26.57,10.31,4.46
2,AirPods Weekly Units,Holt's Damped Trend,28.24,10.96,1.10
3,AirPods Weekly Units,Ridge Regression,32.68,12.68,18.20
4,AirPods Weekly Units,Linear Regression,66.41,25.78,54.06



===== FORECAST BENCHMARK VALIDATION =====
Original models compared: 5 - PASS
Best model: Naive Forecast
Best WAPE: 8.60%
Naive 8.60% benchmark: PASS
Nonnegative, nonmissing WAPE values: PASS

===== AVAILABLE PREDICTION / BACKTEST FILES =====
demand_sensing_holdout_predictions_2019.csv
demand_sensing_model_comparison_2019.csv
external_signal_backtest_predictions_2019.csv
external_signal_model_comparison_2019.csv
forecast_lead_time_baseline_2019.csv
forecast_multihorizon_performance_2019.csv
forecast_multihorizon_predictions_2019.csv
forecast_stability_summary_2019.csv
forecast_uncertainty_summary_2019.csv
model_performance_rolling_2019.csv
model_predictions_rolling_2019.csv
purchase_intent_model_leaderboard.csv

===== STEP 3 COMPLETE =====
Stored forecast performance reconciliation: PASS
Next: Independent forecast-error recalculation.


In [5]:
# STEP 4: INDEPENDENT FORECAST ERROR VALIDATION

import pandas as pd
import numpy as np

print("===== INDEPENDENT FORECAST VALIDATION =====")

# 1. Load prediction-level evidence
prediction_path = (
    PROCESSED_DIR / "model_predictions_rolling_2019.csv"
)

assert prediction_path.exists(), (
    f"Missing rolling predictions: {prediction_path}"
)

predictions_df = pd.read_csv(prediction_path)

print("\n===== PREDICTION DATA INVENTORY =====")
print("File:", prediction_path.name)
print("Rows:", len(predictions_df))
print("Columns:", predictions_df.columns.tolist())

print("\n===== PREDICTION SAMPLE =====")
display(predictions_df.head())

# 2. Identify required fields without assuming exact spelling
def find_column(df, candidates):
    column_lookup = {
        str(col).strip().lower(): col
        for col in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()
        if key in column_lookup:
            return column_lookup[key]

    return None

model_col = find_column(predictions_df, [
    "Model", "Model_Name"
])

actual_col = find_column(predictions_df, [
    "Actual",
    "Actual_Units",
    "Actual_Units_Sold",
    "Actual_Demand",
    "Units_Sold",
    "y_true"
])

forecast_col = find_column(predictions_df, [
    "Prediction",
    "Predicted",
    "Predicted_Units",
    "Predicted_Demand",
    "Forecast",
    "Forecast_Units",
    "Forecasted_Units",
    "y_pred"
])

print("\n===== DETECTED PREDICTION FIELDS =====")
print("Model:", model_col)
print("Actual:", actual_col)
print("Forecast:", forecast_col)

if any(
    col is None
    for col in [model_col, actual_col, forecast_col]
):
    print("\nFIELD MAPPING NEEDED")
    print("Review the columns and sample above.")
    print("No validation result has been claimed.")
else:
    # 3. Prepare the raw actual/forecast records
    backtest = predictions_df[
        [model_col, actual_col, forecast_col]
    ].copy()

    backtest.columns = [
        "Model",
        "Actual",
        "Forecast"
    ]

    backtest["Actual"] = pd.to_numeric(
        backtest["Actual"], errors="raise"
    )

    backtest["Forecast"] = pd.to_numeric(
        backtest["Forecast"], errors="raise"
    )

    assert backtest[
        ["Actual", "Forecast"]
    ].notna().all().all()

    assert np.isfinite(
        backtest[["Actual", "Forecast"]].to_numpy()
    ).all()

    assert (backtest["Actual"] >= 0).all()

    # 4. Calculate forecast errors independently
    backtest["Absolute_Error"] = (
        backtest["Forecast"] - backtest["Actual"]
    ).abs()

    # Convention: positive bias = overforecast
    backtest["Signed_Error"] = (
        backtest["Forecast"] - backtest["Actual"]
    )

    calculated = (
        backtest.groupby("Model", as_index=False)
        .agg(
            Forecasts=("Actual", "size"),
            Total_Actual=("Actual", "sum"),
            Total_Absolute_Error=("Absolute_Error", "sum"),
            MAE=("Absolute_Error", "mean"),
            Bias=("Signed_Error", "mean")
        )
    )

    assert (calculated["Total_Actual"] > 0).all()

    calculated["WAPE_Pct"] = (
        calculated["Total_Absolute_Error"]
        / calculated["Total_Actual"]
        * 100
    )

    calculated = calculated.sort_values(
        "WAPE_Pct"
    ).reset_index(drop=True)

    print("\n===== INDEPENDENTLY RECALCULATED METRICS =====")

    display(
        calculated[
            ["Model", "Forecasts", "MAE", "WAPE_Pct", "Bias"]
        ].round(2)
    )

    # 5. Compare to the five published AirPods model results
    reported = airpods_models[
        ["Model", "MAE", "WAPE_Pct", "Bias"]
    ].copy()

    for metric in ["MAE", "WAPE_Pct", "Bias"]:
        reported[metric] = pd.to_numeric(
            reported[metric], errors="raise"
        )

    comparison = calculated.merge(
        reported,
        on="Model",
        suffixes=("_Calculated", "_Reported"),
        how="outer",
        indicator=True
    )

    print("\n===== REPORTED VS RECALCULATED =====")

    for metric in ["MAE", "WAPE_Pct", "Bias"]:
        comparison[f"{metric}_Difference"] = (
            comparison[f"{metric}_Calculated"]
            - comparison[f"{metric}_Reported"]
        )

    display(comparison.round(4))

    # Allow for published metrics rounded to two decimals
    all_models_match = (
        len(comparison) == 5
        and (comparison["_merge"] == "both").all()
    )

    numeric_match = False

    if all_models_match:
        numeric_match = all(
            comparison[f"{metric}_Difference"]
            .abs().le(0.011).all()
            for metric in ["MAE", "WAPE_Pct", "Bias"]
        )

    print("\n===== FINAL BACKTEST VALIDATION =====")
    print(
        "All five models matched:",
        "PASS" if all_models_match else "REVIEW"
    )
    print(
        "Recalculated metrics agree:",
        "PASS" if numeric_match else "REVIEW"
    )

    if all_models_match and numeric_match:
        print("\n===== STEP 4 COMPLETE: PASS =====")
        print("Forecast metrics independently verified.")
    else:
        print("\n===== STEP 4 REQUIRES REVIEW =====")
        print("Inspect field mappings, model names,")
        print("evaluation periods, and bias conventions.")


===== INDEPENDENT FORECAST VALIDATION =====

===== PREDICTION DATA INVENTORY =====
File: model_predictions_rolling_2019.csv
Rows: 19
Columns: ['Week_Ending', 'Actual_Units', 'Naive_Forecast', 'MA4_Forecast', 'Holt_Forecast', 'Linear_Trend_Forecast', 'Ridge_Forecast']

===== PREDICTION SAMPLE =====


,Week_Ending,Actual_Units,Naive_Forecast,MA4_Forecast,Holt_Forecast,Linear_Trend_Forecast,Ridge_Forecast
0,2019-06-02,263,312,316.50,343.711872,374.457895,322.155958
1,2019-06-09,267,263,302.75,331.210438,361.814286,304.918822
2,2019-06-16,275,267,298.25,270.868345,351.714286,303.440797
3,2019-06-23,265,275,279.25,262.932684,344.387352,294.371375
4,2019-06-30,283,265,267.50,255.182520,336.336957,287.282947



===== DETECTED PREDICTION FIELDS =====
Model: None
Actual: Actual_Units
Forecast: None

FIELD MAPPING NEEDED
Review the columns and sample above.
No validation result has been claimed.


In [6]:
# STEP 4: INDEPENDENT FORECAST ERROR VALIDATION
# Corrected for the wide-format rolling prediction file

import pandas as pd
import numpy as np

print("===== INDEPENDENT FORECAST VALIDATION =====")

# 1. Load existing rolling-origin predictions
prediction_path = (
    PROCESSED_DIR / "model_predictions_rolling_2019.csv"
)

assert prediction_path.exists(), (
    f"Prediction file missing: {prediction_path}"
)

predictions_df = pd.read_csv(prediction_path)

print("\n===== SOURCE PREDICTION DATA =====")
print("File:", prediction_path.name)
print("Rows:", len(predictions_df))
print("Columns:", predictions_df.columns.tolist())

# 2. Define the exact mapping from source fields to model names
model_mapping = {
    "Naive_Forecast": "Naive Forecast",
    "MA4_Forecast": "4-Week Moving Average",
    "Holt_Forecast": "Holt's Damped Trend",
    "Linear_Trend_Forecast": "Linear Regression",
    "Ridge_Forecast": "Ridge Regression"
}

required_columns = [
    "Week_Ending",
    "Actual_Units",
    *model_mapping.keys()
]

missing_columns = [
    col for col in required_columns
    if col not in predictions_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# 3. Validate the original 19-week prediction window
assert len(predictions_df) == 19, (
    f"Expected 19 evaluation weeks, found {len(predictions_df)}."
)

predictions_df["Week_Ending"] = pd.to_datetime(
    predictions_df["Week_Ending"],
    errors="raise"
)

assert predictions_df["Week_Ending"].notna().all()
assert predictions_df["Week_Ending"].is_unique

for col in ["Actual_Units", *model_mapping.keys()]:
    predictions_df[col] = pd.to_numeric(
        predictions_df[col],
        errors="raise"
    )

assert predictions_df[
    ["Actual_Units", *model_mapping.keys()]
].notna().all().all()

assert np.isfinite(
    predictions_df[
        ["Actual_Units", *model_mapping.keys()]
    ].to_numpy()
).all()

assert (predictions_df["Actual_Units"] >= 0).all()

print("\n===== PREDICTION INTEGRITY =====")
print("19 rolling evaluation weeks: PASS")
print("Five prediction columns: PASS")
print("No missing predictions: PASS")
print("No duplicate evaluation dates: PASS")

# 4. Reshape wide predictions into long format
long_predictions = predictions_df.melt(
    id_vars=["Week_Ending", "Actual_Units"],
    value_vars=list(model_mapping.keys()),
    var_name="Source_Model",
    value_name="Forecast"
)

long_predictions["Model"] = (
    long_predictions["Source_Model"]
    .map(model_mapping)
)

long_predictions = long_predictions.rename(
    columns={"Actual_Units": "Actual"}
)

long_predictions = long_predictions[
    ["Week_Ending", "Model", "Actual", "Forecast"]
].sort_values(
    ["Model", "Week_Ending"]
).reset_index(drop=True)

assert len(long_predictions) == 95, (
    "Expected 19 weeks x 5 models = 95 prediction records."
)

print("\n===== STANDARDIZED PREDICTION DATA =====")
print("Evaluation weeks:", predictions_df.shape[0])
print("Forecasting models:", len(model_mapping))
print("Total prediction records:", len(long_predictions))

display(long_predictions.head(10))

# 5. Independently recalculate prediction errors
long_predictions["Absolute_Error"] = (
    long_predictions["Forecast"]
    - long_predictions["Actual"]
).abs()

# Positive bias represents overforecasting
long_predictions["Signed_Error"] = (
    long_predictions["Forecast"]
    - long_predictions["Actual"]
)

calculated = (
    long_predictions
    .groupby("Model", as_index=False)
    .agg(
        Forecasts=("Actual", "size"),
        Total_Actual=("Actual", "sum"),
        Total_Absolute_Error=("Absolute_Error", "sum"),
        MAE=("Absolute_Error", "mean"),
        Bias=("Signed_Error", "mean")
    )
)

assert (calculated["Total_Actual"] > 0).all()
assert (calculated["Forecasts"] == 19).all()

calculated["WAPE_Pct"] = (
    calculated["Total_Absolute_Error"]
    / calculated["Total_Actual"]
    * 100
)

calculated = calculated.sort_values(
    "WAPE_Pct"
).reset_index(drop=True)

print("\n===== INDEPENDENTLY CALCULATED RESULTS =====")

display(
    calculated[
        ["Model", "Forecasts", "MAE", "WAPE_Pct", "Bias"]
    ].round(2)
)

# 6. Load previously verified reporting results from Step 3
reported = airpods_models[
    ["Model", "MAE", "WAPE_Pct", "Bias"]
].copy()

for metric in ["MAE", "WAPE_Pct", "Bias"]:
    reported[metric] = pd.to_numeric(
        reported[metric],
        errors="raise"
    )

# 7. Compare independent results with reported results
comparison = calculated.merge(
    reported,
    on="Model",
    how="outer",
    suffixes=("_Calculated", "_Reported"),
    indicator=True
)

for metric in ["MAE", "WAPE_Pct", "Bias"]:
    comparison[f"{metric}_Difference"] = (
        comparison[f"{metric}_Calculated"]
        - comparison[f"{metric}_Reported"]
    )

print("\n===== INDEPENDENT VS REPORTED COMPARISON =====")

display(
    comparison[
        [
            "Model",
            "MAE_Calculated",
            "MAE_Reported",
            "WAPE_Pct_Calculated",
            "WAPE_Pct_Reported",
            "Bias_Calculated",
            "Bias_Reported"
        ]
    ].round(3)
)

# 8. Validate all model calculations
models_match = (
    len(comparison) == 5
    and (comparison["_merge"] == "both").all()
)

# Reported metrics were rounded to two decimals
tolerance = 0.011

metrics_match = False

if models_match:
    metrics_match = all(
        comparison[f"{metric}_Difference"]
        .abs()
        .le(tolerance)
        .all()
        for metric in ["MAE", "WAPE_Pct", "Bias"]
    )

best_model = calculated.iloc[0]

benchmark_match = (
    best_model["Model"] == "Naive Forecast"
    and abs(best_model["WAPE_Pct"] - 8.60) <= tolerance
)

print("\n===== FINAL VALIDATION RESULTS =====")

print(
    "All five models reconciled:",
    "PASS" if models_match else "REVIEW"
)

print(
    "MAE, WAPE, and bias calculations:",
    "PASS" if metrics_match else "REVIEW"
)

print(
    "Naive forecast is best:",
    "PASS" if benchmark_match else "REVIEW"
)

print(
    "Best model:", best_model["Model"]
)

print(
    "Best WAPE:", f"{best_model['WAPE_Pct']:.2f}%"
)

# 9. Final status
if models_match and metrics_match and benchmark_match:
    print("\n===== STEP 4 COMPLETE: PASS =====")
    print("All 95 rolling predictions validated.")
    print("Forecast performance independently reproduced.")
    print("Original source files remain unchanged.")
else:
    print("\n===== STEP 4 REQUIRES REVIEW =====")
    print("Inspect the comparison table before proceeding.")


===== INDEPENDENT FORECAST VALIDATION =====

===== SOURCE PREDICTION DATA =====
File: model_predictions_rolling_2019.csv
Rows: 19
Columns: ['Week_Ending', 'Actual_Units', 'Naive_Forecast', 'MA4_Forecast', 'Holt_Forecast', 'Linear_Trend_Forecast', 'Ridge_Forecast']

===== PREDICTION INTEGRITY =====
19 rolling evaluation weeks: PASS
Five prediction columns: PASS
No missing predictions: PASS
No duplicate evaluation dates: PASS

===== STANDARDIZED PREDICTION DATA =====
Evaluation weeks: 19
Forecasting models: 5
Total prediction records: 95


,Week_Ending,Model,Actual,Forecast
0,2019-06-02,4-Week Moving Average,263,316.50
1,2019-06-09,4-Week Moving Average,267,302.75
2,2019-06-16,4-Week Moving Average,275,298.25
3,2019-06-23,4-Week Moving Average,265,279.25
4,2019-06-30,4-Week Moving Average,283,267.50
5,2019-07-07,4-Week Moving Average,284,272.50
6,2019-07-14,4-Week Moving Average,295,276.75
7,2019-07-21,4-Week Moving Average,262,281.75
8,2019-07-28,4-Week Moving Average,267,281.00
9,2019-08-04,4-Week Moving Average,227,277.00



===== INDEPENDENTLY CALCULATED RESULTS =====


,Model,Forecasts,MAE,WAPE_Pct,Bias
0,Naive Forecast,19,22.16,8.60,-3.42
1,4-Week Moving Average,19,26.57,10.31,4.46
2,Holt's Damped Trend,19,28.24,10.96,1.10
3,Ridge Regression,19,32.68,12.68,18.20
4,Linear Regression,19,66.41,25.78,54.06



===== INDEPENDENT VS REPORTED COMPARISON =====


,Model,MAE_Calculated,MAE_Reported,WAPE_Pct_Calculated,WAPE_Pct_Reported,Bias_Calculated,Bias_Reported
0,4-Week Moving Average,26.566,26.566,10.312,10.312,4.461,4.461
1,Holt's Damped Trend,28.241,28.241,10.962,10.962,1.095,1.095
2,Linear Regression,66.412,66.412,25.778,25.778,54.064,54.064
3,Naive Forecast,22.158,22.158,8.601,8.601,-3.421,-3.421
4,Ridge Regression,32.679,32.679,12.684,12.684,18.201,18.201



===== FINAL VALIDATION RESULTS =====
All five models reconciled: PASS
MAE, WAPE, and bias calculations: PASS
Naive forecast is best: PASS
Best model: Naive Forecast
Best WAPE: 8.60%

===== STEP 4 COMPLETE: PASS =====
All 95 rolling predictions validated.
Forecast performance independently reproduced.
Original source files remain unchanged.


In [7]:
# STEP 5: EXTERNAL DEMAND SIGNAL VALIDATION
# Read-only audit of model results, signal alignment, and stability

import pandas as pd
import duckdb
import numpy as np

print("===== EXTERNAL DEMAND SIGNAL VALIDATION =====")

# --------------------------------------------------
# 1. Define existing project files
# --------------------------------------------------

model_path = (
    PROCESSED_DIR
    / "external_signal_model_comparison_2019.csv"
)

stability_path = (
    PROCESSED_DIR
    / "external_signal_stability_2019.csv"
)

dashboard_path = (
    DASHBOARD_DIR
    / "executive_v1"
    / "dashboard_v_external_signal_model_comparison.csv"
)

warehouse_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

required_files = {
    "Model comparison": model_path,
    "Stability analysis": stability_path,
    "Tableau model export": dashboard_path,
    "DuckDB warehouse": warehouse_path
}

print("\n===== FILE AVAILABILITY =====")

for name, path in required_files.items():
    assert path.exists(), f"Missing {name}: {path}"
    print(f"{name}: PASS")

# --------------------------------------------------
# 2. Load processed external-signal results
# --------------------------------------------------

signal_models = pd.read_csv(model_path)
signal_stability = pd.read_csv(stability_path)

print("\n===== PROCESSED DATA INVENTORY =====")

print("Model comparison rows:", len(signal_models))
print("Model comparison fields:", signal_models.columns.tolist())

print("\nStability rows:", len(signal_stability))
print("Stability fields:", signal_stability.columns.tolist())

print("\n===== SOURCE MODEL RESULTS =====")
display(signal_models)

print("\n===== SIGNAL STABILITY SAMPLE =====")
display(signal_stability.head(10))

# --------------------------------------------------
# 3. Load DuckDB reporting views independently
# --------------------------------------------------

con = duckdb.connect(
    str(warehouse_path),
    read_only=True
)

try:
    warehouse_models = con.execute("""
        SELECT *
        FROM reporting.v_external_signal_model_comparison
    """).df()

    aligned_signals = con.execute("""
        SELECT *
        FROM reporting.v_external_signal_aligned_demand
    """).df()

finally:
    con.close()

dashboard_models = pd.read_csv(dashboard_path)

print("\n===== REPORTING DATA INVENTORY =====")

print("Warehouse model rows:", len(warehouse_models))
print("Tableau model rows:", len(dashboard_models))
print("Aligned demand weeks:", len(aligned_signals))
print("Aligned signal columns:", aligned_signals.columns.tolist())

# --------------------------------------------------
# 4. Reconcile Tableau and DuckDB model results
# --------------------------------------------------

assert set(warehouse_models.columns) == set(
    dashboard_models.columns
), "Tableau and DuckDB model columns differ."

def normalize_model_report(df):
    result = df.copy()

    # Normalize numeric representations where possible
    for col in result.columns:
        if pd.api.types.is_datetime64_any_dtype(result[col]):
            result[col] = result[col].astype(str)

    result = result[sorted(result.columns)]

    # Sort using stable text representations of rows
    sort_key = result.astype(str).agg(
        "|".join, axis=1
    )

    return (
        result.assign(_sort_key=sort_key)
        .sort_values("_sort_key")
        .drop(columns="_sort_key")
        .reset_index(drop=True)
    )

pd.testing.assert_frame_equal(
    normalize_model_report(dashboard_models),
    normalize_model_report(warehouse_models),
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-6
)

print("\n===== TABLEAU / DUCKDB RECONCILIATION =====")
print("Model reporting records match: PASS")

# --------------------------------------------------
# 5. Validate model counts and reported metrics
# --------------------------------------------------

assert len(signal_models) == 3, (
    f"Expected 3 external-signal model results, "
    f"found {len(signal_models)}."
)

assert len(warehouse_models) == 3, (
    "Expected 3 warehouse model results."
)

assert len(aligned_signals) == 51, (
    f"Expected 51 aligned demand weeks, "
    f"found {len(aligned_signals)}."
)

print("\n===== SIGNAL MODEL VALIDATION =====")
print("3 external-signal models: PASS")
print("51 aligned demand weeks: PASS")

if "WAPE_Pct" in signal_models.columns:

    signal_models["WAPE_Pct"] = pd.to_numeric(
        signal_models["WAPE_Pct"],
        errors="raise"
    )

    assert signal_models["WAPE_Pct"].notna().all()
    assert np.isfinite(signal_models["WAPE_Pct"]).all()
    assert (signal_models["WAPE_Pct"] >= 0).all()

    print("Valid WAPE metrics: PASS")

    if "Model" in signal_models.columns:

        ranked = signal_models.sort_values(
            "WAPE_Pct"
        ).reset_index(drop=True)

        print("\n===== EXTERNAL MODEL RANKING =====")

        display(
            ranked[
                [
                    col for col in [
                        "Model",
                        "Forecasts",
                        "MAE",
                        "WAPE_Pct",
                        "Bias"
                    ]
                    if col in ranked.columns
                ]
            ].round(3)
        )

        print(
            "Lowest WAPE model:",
            ranked.iloc[0]["Model"]
        )

        print(
            "Lowest external-model WAPE:",
            f"{ranked.iloc[0]['WAPE_Pct']:.2f}%"
        )

else:
    print("WAPE_Pct not found in processed file.")
    print("Review source column definitions above.")

# --------------------------------------------------
# 6. Inspect signal alignment and coverage
# --------------------------------------------------

print("\n===== SIGNAL ALIGNMENT SAMPLE =====")

display(aligned_signals.head(10))

print("\n===== SIGNAL COVERAGE =====")

coverage = pd.DataFrame({
    "Field": aligned_signals.columns,
    "Records": len(aligned_signals),
    "Nonmissing": aligned_signals.notna().sum().values,
    "Missing": aligned_signals.isna().sum().values
})

coverage["Coverage_Pct"] = (
    coverage["Nonmissing"]
    / coverage["Records"]
    * 100
)

display(coverage.round(2))

# Missing values can be legitimate for lagged
# Google Trends signals at dataset boundaries.
# Do not treat them as automatically invalid.

# --------------------------------------------------
# 7. Final audit summary
# --------------------------------------------------

audit = pd.DataFrame([
    {
        "Validation": "External model records",
        "Expected": 3,
        "Observed": len(signal_models),
        "Status": "PASS"
    },
    {
        "Validation": "Tableau / DuckDB model parity",
        "Expected": "Match",
        "Observed": "Match",
        "Status": "PASS"
    },
    {
        "Validation": "Aligned historical weeks",
        "Expected": 51,
        "Observed": len(aligned_signals),
        "Status": "PASS"
    },
    {
        "Validation": "Stability file available",
        "Expected": "Yes",
        "Observed": "Yes",
        "Status": "PASS"
    }
])

print("\n===== EXTERNAL SIGNAL AUDIT =====")
display(audit)

print("\n===== STEP 5 INITIAL VALIDATION COMPLETE =====")
print("Source availability and reporting parity: PASS")
print("Model-ranking and stability interpretation:")
print("Review tables before concluding signal benefit.")
print("Original source files remain unchanged.")


===== EXTERNAL DEMAND SIGNAL VALIDATION =====

===== FILE AVAILABILITY =====
Model comparison: PASS
Stability analysis: PASS
Tableau model export: PASS
DuckDB warehouse: PASS

===== PROCESSED DATA INVENTORY =====
Model comparison rows: 3
Model comparison fields: ['Model', 'Forecasts', 'MAE', 'WAPE_Pct', 'Bias', 'WAPE_Gap_vs_Naive_pp']

Stability rows: 6
Stability fields: ['Period', 'Model', 'Forecasts', 'WAPE_Pct']

===== SOURCE MODEL RESULTS =====


,Model,Forecasts,MAE,WAPE_Pct,Bias,WAPE_Gap_vs_Naive_pp
0,Naive Forecast,19,22.157895,8.600613,-3.421053,0.000000
1,Ridge - Sales + Google Trends,19,25.548146,9.916543,9.755460,1.315930
2,Ridge - Sales History,19,28.035306,10.881937,12.981336,2.281324



===== SIGNAL STABILITY SAMPLE =====


,Period,Model,Forecasts,WAPE_Pct
0,First 9 Evaluation Weeks,Naive,9,5.648111
1,First 9 Evaluation Weeks,Sales_Ridge,9,6.626247
2,First 9 Evaluation Weeks,Trends_Ridge,9,6.975139
3,Last 10 Evaluation Weeks,Naive,10,11.585867
4,Last 10 Evaluation Weeks,Sales_Ridge,10,15.184835
5,Last 10 Evaluation Weeks,Trends_Ridge,10,12.890575



===== REPORTING DATA INVENTORY =====
Warehouse model rows: 3
Tableau model rows: 3
Aligned demand weeks: 51
Aligned signal columns: ['Week_Ending', 'Units_Sold', 'Forecast_Origin', 'Overlapping_Week_Search_Interest', 'Prior_Completed_Search_Interest', 'Boundary_Week_Flag']

===== TABLEAU / DUCKDB RECONCILIATION =====
Model reporting records match: PASS

===== SIGNAL MODEL VALIDATION =====
3 external-signal models: PASS
51 aligned demand weeks: PASS
Valid WAPE metrics: PASS

===== EXTERNAL MODEL RANKING =====


,Model,Forecasts,MAE,WAPE_Pct,Bias
0,Naive Forecast,19,22.158,8.601,-3.421
1,Ridge - Sales + Google Trends,19,25.548,9.917,9.755
2,Ridge - Sales History,19,28.035,10.882,12.981


... (3 outputs left out)

,Field,Records,Nonmissing,Missing,Coverage_Pct
0,Week_Ending,51,51,0,100.0
1,Units_Sold,51,51,0,100.0
2,Forecast_Origin,51,51,0,100.0
3,Overlapping_Week_Search_Interest,51,51,0,100.0
4,Prior_Completed_Search_Interest,51,51,0,100.0
5,Boundary_Week_Flag,51,51,0,100.0



===== EXTERNAL SIGNAL AUDIT =====


,Validation,Expected,Observed,Status
0,External model records,3,3,PASS
1,Tableau / DuckDB model parity,Match,Match,PASS
2,Aligned historical weeks,51,51,PASS
3,Stability file available,Yes,Yes,PASS



===== STEP 5 INITIAL VALIDATION COMPLETE =====
Source availability and reporting parity: PASS
Model-ranking and stability interpretation:
Review tables before concluding signal benefit.
Original source files remain unchanged.


In [8]:
# STEP 6: INDEPENDENT INVENTORY SIMULATION VALIDATION

import pandas as pd
import numpy as np
import duckdb

print("===== INVENTORY SIMULATION VALIDATION =====")

# --------------------------------------------------
# 1. Locate existing validated reporting sources
# --------------------------------------------------

warehouse_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

dashboard_path = (
    DASHBOARD_DIR
    / "executive_v1"
    / "dashboard_v_inventory_scenario_performance.csv"
)

assert warehouse_path.exists(), "Warehouse not found."
assert dashboard_path.exists(), "Dashboard export not found."

# --------------------------------------------------
# 2. Load weekly and scenario-level inventory data
# --------------------------------------------------

con = duckdb.connect(
    str(warehouse_path),
    read_only=True
)

try:
    weekly = con.execute("""
        SELECT *
        FROM reporting.v_inventory_weekly
        ORDER BY Scenario, Week
    """).df()

    reported = con.execute("""
        SELECT *
        FROM reporting.v_inventory_scenario_performance
        ORDER BY Scenario
    """).df()

finally:
    con.close()

dashboard = pd.read_csv(dashboard_path)

print("\n===== INVENTORY DATA INVENTORY =====")
print("Weekly simulation records:", len(weekly))
print("Scenario summary records:", len(reported))
print("Tableau export records:", len(dashboard))

print("\nWeekly fields:", weekly.columns.tolist())
print("Scenario fields:", reported.columns.tolist())

print("\n===== WEEKLY SIMULATION SAMPLE =====")
display(weekly.head(6))

# --------------------------------------------------
# 3. Validate required fields
# --------------------------------------------------

weekly_required = [
    "Scenario",
    "Week",
    "Demand_Units",
    "Fulfilled_Units",
    "Shortage_Units",
    "Ending_Inventory"
]

summary_required = [
    "Scenario",
    "Total_Demand",
    "Total_Fulfilled",
    "Total_Shortage",
    "Fill_Rate_Pct",
    "Shortage_Weeks"
]

assert set(weekly_required).issubset(weekly.columns), (
    "Required weekly columns missing."
)

assert set(summary_required).issubset(reported.columns), (
    "Required scenario summary columns missing."
)

assert len(weekly) == 12, (
    f"Expected 12 weekly records, found {len(weekly)}."
)

assert len(reported) == 3, (
    f"Expected 3 scenario records, found {len(reported)}."
)

# --------------------------------------------------
# 4. Check weekly inventory arithmetic
# --------------------------------------------------

numeric_fields = [
    "Demand_Units",
    "Fulfilled_Units",
    "Shortage_Units",
    "Ending_Inventory"
]

for field in numeric_fields:
    weekly[field] = pd.to_numeric(
        weekly[field], errors="raise"
    )

assert weekly[numeric_fields].notna().all().all()
assert np.isfinite(weekly[numeric_fields].to_numpy()).all()

assert (weekly[numeric_fields] >= 0).all().all(), (
    "Negative inventory-related quantities found."
)

weekly["Demand_Balance_Difference"] = (
    weekly["Demand_Units"]
    - weekly["Fulfilled_Units"]
    - weekly["Shortage_Units"]
)

demand_balance_pass = np.allclose(
    weekly["Demand_Balance_Difference"],
    0,
    atol=1e-6
)

print("\n===== WEEKLY ARITHMETIC =====")
print(
    "Demand = Fulfilled + Shortage:",
    "PASS" if demand_balance_pass else "REVIEW"
)

# --------------------------------------------------
# 5. Independently aggregate scenario performance
# --------------------------------------------------

calculated = (
    weekly.groupby("Scenario", as_index=False)
    .agg(
        Total_Demand=("Demand_Units", "sum"),
        Total_Fulfilled=("Fulfilled_Units", "sum"),
        Total_Shortage=("Shortage_Units", "sum"),
        Shortage_Weeks=(
            "Shortage_Units",
            lambda x: int((x > 0).sum())
        ),
        Weeks=("Week", "nunique")
    )
)

assert (calculated["Total_Demand"] > 0).all()

calculated["Fill_Rate_Pct"] = (
    calculated["Total_Fulfilled"]
    / calculated["Total_Demand"]
    * 100
)

print("\n===== INDEPENDENT SCENARIO RESULTS =====")

display(calculated.round(2))

# --------------------------------------------------
# 6. Compare recalculated values against DuckDB
# --------------------------------------------------

metrics = [
    "Total_Demand",
    "Total_Fulfilled",
    "Total_Shortage",
    "Shortage_Weeks",
    "Fill_Rate_Pct"
]

comparison = calculated.merge(
    reported[["Scenario", *metrics]],
    on="Scenario",
    how="outer",
    suffixes=("_Calculated", "_Reported"),
    indicator=True
)

print("\n===== CALCULATED VS REPORTED =====")

metrics_match = (
    len(comparison) == 3
    and (comparison["_merge"] == "both").all()
)

for metric in metrics:
    calculated_field = f"{metric}_Calculated"
    reported_field = f"{metric}_Reported"

    comparison[reported_field] = pd.to_numeric(
        comparison[reported_field],
        errors="raise"
    )

    comparison[f"{metric}_Difference"] = (
        comparison[calculated_field]
        - comparison[reported_field]
    )

    metric_pass = comparison[
        f"{metric}_Difference"
    ].abs().le(0.011).all()

    metrics_match = metrics_match and metric_pass

display(
    comparison[
        ["Scenario", *[
            f"{metric}_Difference"
            for metric in metrics
        ]]
    ].round(4)
)

print(
    "Scenario metric reconciliation:",
    "PASS" if metrics_match else "REVIEW"
)

# --------------------------------------------------
# 7. Validate scenario-level expected outcomes
# --------------------------------------------------

expected = pd.DataFrame([
    {
        "Scenario": "Baseline",
        "Expected_Shortage": 0,
        "Expected_Fill_Rate": 100.00
    },
    {
        "Scenario": "Demand Surge",
        "Expected_Shortage": 276,
        "Expected_Fill_Rate": 89.29
    },
    {
        "Scenario": "Supplier Delay",
        "Expected_Shortage": 660,
        "Expected_Fill_Rate": 67.96
    }
])

outcomes = calculated.merge(
    expected,
    on="Scenario",
    how="outer",
    indicator=True
)

expected_match = (
    len(outcomes) == 3
    and (outcomes["_merge"] == "both").all()
    and np.allclose(
        outcomes["Total_Shortage"],
        outcomes["Expected_Shortage"],
        atol=1e-6
    )
    and np.allclose(
        outcomes["Fill_Rate_Pct"],
        outcomes["Expected_Fill_Rate"],
        atol=0.02
    )
)

print("\n===== EXPECTED SCENARIO OUTCOMES =====")

display(
    outcomes[
        [
            "Scenario",
            "Total_Shortage",
            "Expected_Shortage",
            "Fill_Rate_Pct",
            "Expected_Fill_Rate"
        ]
    ].round(2)
)

print(
    "Expected simulation outcomes:",
    "PASS" if expected_match else "REVIEW"
)

# --------------------------------------------------
# 8. Reconcile Tableau export with DuckDB reporting
# --------------------------------------------------

common_columns = sorted(
    set(reported.columns).intersection(dashboard.columns)
)

assert "Scenario" in common_columns

reported_check = (
    reported[common_columns]
    .sort_values("Scenario")
    .reset_index(drop=True)
)

dashboard_check = (
    dashboard[common_columns]
    .sort_values("Scenario")
    .reset_index(drop=True)
)

dashboard_match = True

try:
    pd.testing.assert_frame_equal(
        reported_check,
        dashboard_check,
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=0.011
    )
except AssertionError as error:
    dashboard_match = False
    print("Dashboard mismatch detail:", str(error)[:500])

print("\n===== DASHBOARD RECONCILIATION =====")
print(
    "Tableau vs DuckDB:",
    "PASS" if dashboard_match else "REVIEW"
)

# --------------------------------------------------
# 9. Final results
# --------------------------------------------------

checks = {
    "Weekly demand arithmetic": demand_balance_pass,
    "Scenario totals independently reproduced": metrics_match,
    "Expected scenario outcomes": expected_match,
    "Tableau reporting parity": dashboard_match
}

print("\n===== STEP 6 VALIDATION SUMMARY =====")

for label, passed in checks.items():
    print(
        f"{label}:",
        "PASS" if passed else "REVIEW"
    )

if all(checks.values()):
    print("\n===== STEP 6 COMPLETE: PASS =====")
    print("Inventory scenario metrics independently validated.")
else:
    print("\n===== STEP 6 REQUIRES REVIEW =====")
    print("Inspect any discrepancies before proceeding.")

print("Read-only validation: source files unchanged.")


===== INVENTORY SIMULATION VALIDATION =====

===== INVENTORY DATA INVENTORY =====
Weekly simulation records: 12
Scenario summary records: 3
Tableau export records: 3

Weekly fields: ['Scenario', 'Week', 'Forecast_Week', 'Beginning_Inventory', 'Inbound_Receipts', 'Demand_Units', 'Fulfilled_Units', 'Shortage_Units', 'Ending_Inventory', 'Weekly_Fill_Rate_Pct']
Scenario fields: ['Scenario', 'Total_Demand', 'Total_Fulfilled', 'Total_Shortage', 'Receipts_In_Horizon', 'Shortage_Weeks', 'Ending_Inventory', 'Fill_Rate_Pct']

===== WEEKLY SIMULATION SAMPLE =====


,Scenario,Week,Forecast_Week,Beginning_Inventory,Inbound_Receipts,Demand_Units,Fulfilled_Units,Shortage_Units,Ending_Inventory,Weekly_Fill_Rate_Pct
0,Baseline,1,2020-01-05,1400,0,515,515,0,885,100.0
1,Baseline,2,2020-01-12,885,0,515,515,0,370,100.0
2,Baseline,3,2020-01-19,370,900,515,515,0,755,100.0
3,Baseline,4,2020-01-26,755,0,515,515,0,240,100.0
4,Demand Surge,1,2020-01-05,1400,0,644,644,0,756,100.0
5,Demand Surge,2,2020-01-12,756,0,644,644,0,112,100.0



===== WEEKLY ARITHMETIC =====
Demand = Fulfilled + Shortage: PASS

===== INDEPENDENT SCENARIO RESULTS =====


,Scenario,Total_Demand,Total_Fulfilled,Total_Shortage,Shortage_Weeks,Weeks,Fill_Rate_Pct
0,Baseline,2060,2060,0,0,4,100.00
1,Demand Surge,2576,2300,276,1,4,89.29
2,Supplier Delay,2060,1400,660,2,4,67.96



===== CALCULATED VS REPORTED =====


,Scenario,Total_Demand_Difference,Total_Fulfilled_Difference,Total_Shortage_Difference,Shortage_Weeks_Difference,Fill_Rate_Pct_Difference
0,Baseline,0,0,0,0,0.0
1,Demand Surge,0,0,0,0,0.0
2,Supplier Delay,0,0,0,0,0.0


Scenario metric reconciliation: PASS

===== EXPECTED SCENARIO OUTCOMES =====


,Scenario,Total_Shortage,Expected_Shortage,Fill_Rate_Pct,Expected_Fill_Rate
0,Baseline,0,0,100.00,100.00
1,Demand Surge,276,276,89.29,89.29
2,Supplier Delay,660,660,67.96,67.96


Expected simulation outcomes: PASS

===== DASHBOARD RECONCILIATION =====
Tableau vs DuckDB: PASS

===== STEP 6 VALIDATION SUMMARY =====
Weekly demand arithmetic: PASS
Scenario totals independently reproduced: PASS
Expected scenario outcomes: PASS
Tableau reporting parity: PASS

===== STEP 6 COMPLETE: PASS =====
Inventory scenario metrics independently validated.
Read-only validation: source files unchanged.


In [9]:
# STEP 7: REPLENISHMENT AND ECONOMIC EXPOSURE VALIDATION
# Read-only reconciliation of the seven decision options

import pandas as pd
import numpy as np
import duckdb

print("===== REPLENISHMENT ECONOMICS VALIDATION =====")

warehouse_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

dashboard_path = (
    DASHBOARD_DIR
    / "executive_v1"
    / "dashboard_v_inventory_launch_readiness.csv"
)

assert warehouse_path.exists()
assert dashboard_path.exists()

# 1. Load decision-level and scenario-level reporting data
con = duckdb.connect(str(warehouse_path), read_only=True)

try:
    actions_raw = con.execute("""
        SELECT *
        FROM reporting.v_inventory_launch_readiness
    """).df()

    scenarios = con.execute("""
        SELECT Scenario, Total_Demand, Total_Shortage
        FROM reporting.v_inventory_scenario_performance
    """).df()

    candidates = con.execute("""
        SELECT *
        FROM reporting.v_inventory_candidate_actions
    """).df()

finally:
    con.close()

dashboard_raw = pd.read_csv(dashboard_path)

print("\n===== SOURCE INVENTORY =====")
print("Scenario-action records:", len(actions_raw))
print("Candidate-action records:", len(candidates))
print("Dashboard export records:", len(dashboard_raw))
print("Action fields:", actions_raw.columns.tolist())
print("Candidate fields:", candidates.columns.tolist())

# 2. Resolve field names
# Supports minor variations in column naming
def normalized(name):
    return "".join(c for c in str(name).lower()
                   if c.isalnum())

def find_field(df, expected):
    target = normalized(expected)

    for col in df.columns:
        if normalized(col) == target:
            return col

    for col in df.columns:
        if normalized(col).startswith(target):
            return col

    raise KeyError(
        f"Cannot locate {expected}. "
        f"Available fields: {df.columns.tolist()}"
    )

field_map = {
    "Scenario": "Scenario",
    "Action": "Action",
    "Fill_Rate_Pct": "Fill_Rate_Pct",
    "Total_Shortage": "Total_Shortage",
    "Illustrative_Exposure_USD": "Illustrative_Exposure_USD",
    "Exposure_Reduction_USD": "Exposure_Reduction_vs_No_Action"
}

actions = pd.DataFrame({
    target: actions_raw[find_field(actions_raw, source)]
    for target, source in field_map.items()
})

numeric_cols = [
    "Fill_Rate_Pct",
    "Total_Shortage",
    "Illustrative_Exposure_USD",
    "Exposure_Reduction_USD"
]

for col in numeric_cols:
    actions[col] = pd.to_numeric(actions[col], errors="raise")

assert actions[numeric_cols].notna().all().all()
assert np.isfinite(actions[numeric_cols].to_numpy()).all()
assert len(actions) == 7
assert not actions.duplicated(["Scenario", "Action"]).any()

print("\n===== SEVEN DECISION OPTIONS =====")
display(actions.sort_values(["Scenario", "Action"]).round(2))

# 3. Independently recalculate action fill rates
scenarios["Total_Demand"] = pd.to_numeric(
    scenarios["Total_Demand"], errors="raise"
)

recalculated = actions.merge(
    scenarios[["Scenario", "Total_Demand"]],
    on="Scenario",
    how="left",
    validate="many_to_one"
)

assert recalculated["Total_Demand"].notna().all()
assert (recalculated["Total_Demand"] > 0).all()

recalculated["Calculated_Fill_Rate_Pct"] = (
    (
        recalculated["Total_Demand"]
        - recalculated["Total_Shortage"]
    )
    / recalculated["Total_Demand"]
    * 100
)

fill_rate_pass = np.allclose(
    recalculated["Calculated_Fill_Rate_Pct"],
    recalculated["Fill_Rate_Pct"],
    atol=0.02
)

print("\n===== FILL RATE RECALCULATION =====")
display(
    recalculated[
        [
            "Scenario", "Action",
            "Total_Demand", "Total_Shortage",
            "Fill_Rate_Pct", "Calculated_Fill_Rate_Pct"
        ]
    ].round(2)
)

print("Action fill-rate arithmetic:",
      "PASS" if fill_rate_pass else "REVIEW")

# 4. Reconstruct financial exposure reductions
# from each scenario's No Intervention reference
no_action = actions[
    actions["Action"].astype(str)
    .str.strip().str.casefold()
    .eq("no intervention")
].copy()

assert len(no_action) == 2

references = no_action[
    ["Scenario", "Illustrative_Exposure_USD"]
].rename(columns={
    "Illustrative_Exposure_USD": "No_Action_Exposure_USD"
})

economics = recalculated.merge(
    references,
    on="Scenario",
    how="left",
    validate="many_to_one"
)

assert economics["No_Action_Exposure_USD"].notna().all()

economics["Calculated_Reduction_USD"] = (
    economics["No_Action_Exposure_USD"]
    - economics["Illustrative_Exposure_USD"]
)

economics_pass = np.allclose(
    economics["Calculated_Reduction_USD"],
    economics["Exposure_Reduction_USD"],
    atol=0.02
)

print("\n===== ECONOMIC EXPOSURE RECONCILIATION =====")
display(
    economics[
        [
            "Scenario", "Action",
            "No_Action_Exposure_USD",
            "Illustrative_Exposure_USD",
            "Exposure_Reduction_USD",
            "Calculated_Reduction_USD"
        ]
    ].round(2)
)

print("Exposure reduction arithmetic:",
      "PASS" if economics_pass else "REVIEW")

# 5. Check No Intervention outcomes against Step 6
baseline_check = no_action.merge(
    scenarios,
    on="Scenario",
    validate="one_to_one"
)

baseline_pass = np.allclose(
    baseline_check["Total_Shortage_x"],
    baseline_check["Total_Shortage_y"],
    atol=1e-6
)

print("\n===== NO-INTERVENTION BASELINE =====")
print("Shortages agree with Step 6:",
      "PASS" if baseline_pass else "REVIEW")

# 6. Verify the two executive candidate actions
assert len(candidates) == 2

candidate_scenario = find_field(candidates, "Scenario")
candidate_action = find_field(candidates, "Action")

candidate_keys = candidates[
    [candidate_scenario, candidate_action]
].copy()

candidate_keys.columns = ["Scenario", "Action"]

candidate_match = candidate_keys.merge(
    actions,
    on=["Scenario", "Action"],
    how="left",
    validate="one_to_one",
    indicator=True
)

candidate_pass = (
    (candidate_match["_merge"] == "both").all()
    and (candidate_match["Total_Shortage"] == 0).all()
    and (candidate_match["Fill_Rate_Pct"] >= 98).all()
)

print("\n===== EXECUTIVE CANDIDATE ACTIONS =====")
display(
    candidate_match[
        [
            "Scenario", "Action",
            "Fill_Rate_Pct",
            "Total_Shortage",
            "Illustrative_Exposure_USD",
            "Exposure_Reduction_USD"
        ]
    ].round(2)
)

print("Both candidates meet readiness:",
      "PASS" if candidate_pass else "REVIEW")

# 7. Check Tableau export against DuckDB view
dashboard_match = True

try:
    assert set(dashboard_raw.columns) == set(
        actions_raw.columns
    )

    columns = sorted(actions_raw.columns)

    def sort_report(df):
        return (
            df[columns]
            .sort_values(["Scenario", "Action"])
            .reset_index(drop=True)
        )

    pd.testing.assert_frame_equal(
        sort_report(actions_raw),
        sort_report(dashboard_raw),
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=1e-6
    )

except (AssertionError, KeyError) as error:
    dashboard_match = False
    print("Dashboard comparison detail:", str(error)[:400])

print("\n===== TABLEAU RECONCILIATION =====")
print("Tableau and DuckDB agree:",
      "PASS" if dashboard_match else "REVIEW")

# 8. Confirm the $47,000 modeled result
expedite = economics[
    (economics["Scenario"] == "Supplier Delay")
    & economics["Action"].astype(str)
        .str.contains("Expedite Original", case=False)
]

expedite_pass = (
    len(expedite) == 1
    and np.isclose(
        expedite.iloc[0]["Calculated_Reduction_USD"],
        47000,
        atol=0.02
    )
)

print("\n===== KEY EXECUTIVE RESULT =====")
print("$47,000 modeled exposure reduction:",
      "PASS" if expedite_pass else "REVIEW")

# 9. Summary
checks = {
    "Seven unique scenario-action records": True,
    "Independent fill-rate calculations": fill_rate_pass,
    "Exposure reduction calculations": economics_pass,
    "No-intervention baseline parity": baseline_pass,
    "Two launch-ready candidates": candidate_pass,
    "Tableau / DuckDB parity": dashboard_match,
    "$47,000 modeled exposure reduction": expedite_pass
}

print("\n===== STEP 7 VALIDATION SUMMARY =====")
for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'REVIEW'}")

if all(checks.values()):
    print("\n===== STEP 7 COMPLETE: PASS =====")
    print("Replenishment and exposure calculations reconciled.")
else:
    print("\n===== STEP 7 REQUIRES REVIEW =====")

print("Source datasets remain unchanged.")


===== REPLENISHMENT ECONOMICS VALIDATION =====

===== SOURCE INVENTORY =====
Scenario-action records: 7
Candidate-action records: 2
Dashboard export records: 7
Action fields: ['Scenario', 'Action', 'Total_Shortage', 'Shortage_Weeks', 'Fill_Rate_Pct', 'Readiness_Status', 'Illustrative_Exposure_USD', 'Exposure_Reduction_vs_No_Action_USD', 'Total_Outstanding_Units']
Candidate fields: ['Scenario', 'Action', 'Fill_Rate_Pct', 'Illustrative_Exposure_USD', 'Total_Outstanding_Units']

===== SEVEN DECISION OPTIONS =====


,Scenario,Action,Fill_Rate_Pct,Total_Shortage,Illustrative_Exposure_USD,Exposure_Reduction_USD
1,Demand Surge,Add 276 Units - Week 4,100.00,0,3812.0,16888.0
2,Demand Surge,Add 276 Units - Week 5,89.29,276,20700.0,0.0
0,Demand Surge,No Intervention,89.29,276,20700.0,0.0
4,Supplier Delay,Bridge 660 Units - Week 3,100.00,0,8420.0,41080.0
5,Supplier Delay,Bridge 660 Units - Week 4,92.96,145,19295.0,30205.0
6,Supplier Delay,Expedite Original 900 - Week 3,100.00,0,2500.0,47000.0
3,Supplier Delay,No Intervention,67.96,660,49500.0,0.0



===== FILL RATE RECALCULATION =====


,Scenario,Action,Total_Demand,Total_Shortage,Fill_Rate_Pct,Calculated_Fill_Rate_Pct
0,Demand Surge,No Intervention,2576,276,89.29,89.29
1,Demand Surge,Add 276 Units - Week 4,2576,0,100.00,100.00
2,Demand Surge,Add 276 Units - Week 5,2576,276,89.29,89.29
3,Supplier Delay,No Intervention,2060,660,67.96,67.96
4,Supplier Delay,Bridge 660 Units - Week 3,2060,0,100.00,100.00
5,Supplier Delay,Bridge 660 Units - Week 4,2060,145,92.96,92.96
6,Supplier Delay,Expedite Original 900 - Week 3,2060,0,100.00,100.00


Action fill-rate arithmetic: PASS

===== ECONOMIC EXPOSURE RECONCILIATION =====


,Scenario,Action,No_Action_Exposure_USD,Illustrative_Exposure_USD,Exposure_Reduction_USD,Calculated_Reduction_USD
0,Demand Surge,No Intervention,20700.0,20700.0,0.0,0.0
1,Demand Surge,Add 276 Units - Week 4,20700.0,3812.0,16888.0,16888.0
2,Demand Surge,Add 276 Units - Week 5,20700.0,20700.0,0.0,0.0
3,Supplier Delay,No Intervention,49500.0,49500.0,0.0,0.0
4,Supplier Delay,Bridge 660 Units - Week 3,49500.0,8420.0,41080.0,41080.0
5,Supplier Delay,Bridge 660 Units - Week 4,49500.0,19295.0,30205.0,30205.0
6,Supplier Delay,Expedite Original 900 - Week 3,49500.0,2500.0,47000.0,47000.0


Exposure reduction arithmetic: PASS

===== NO-INTERVENTION BASELINE =====
Shortages agree with Step 6: PASS

===== EXECUTIVE CANDIDATE ACTIONS =====


,Scenario,Action,Fill_Rate_Pct,Total_Shortage,Illustrative_Exposure_USD,Exposure_Reduction_USD
0,Demand Surge,Add 276 Units - Week 4,100.0,0,3812.0,16888.0
1,Supplier Delay,Expedite Original 900 - Week 3,100.0,0,2500.0,47000.0


Both candidates meet readiness: PASS

===== TABLEAU RECONCILIATION =====
Tableau and DuckDB agree: PASS

===== KEY EXECUTIVE RESULT =====
$47,000 modeled exposure reduction: PASS

===== STEP 7 VALIDATION SUMMARY =====
Seven unique scenario-action records: PASS
Independent fill-rate calculations: PASS
Exposure reduction calculations: PASS
No-intervention baseline parity: PASS
Two launch-ready candidates: PASS
Tableau / DuckDB parity: PASS
$47,000 modeled exposure reduction: PASS

===== STEP 7 COMPLETE: PASS =====
Replenishment and exposure calculations reconciled.
Source datasets remain unchanged.


In [10]:
# STEP 8: SENSITIVITY AND DECISION ROBUSTNESS VALIDATION
# Read-only verification of cases, interventions and coverage

import pandas as pd
import numpy as np
import duckdb

print("===== SENSITIVITY ANALYSIS VALIDATION =====")

warehouse_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert warehouse_path.exists()

# 1. Load reporting views and original action definitions
con = duckdb.connect(str(warehouse_path), read_only=True)

try:
    cases = con.execute("""
        SELECT * FROM reporting.v_sensitivity_decision_cases
    """).df()

    evaluations = con.execute("""
        SELECT * FROM reporting.v_sensitivity_intervention_evaluations
    """).df()

    robustness = con.execute("""
        SELECT * FROM reporting.v_sensitivity_action_robustness
    """).df()

    coverage = con.execute("""
        SELECT * FROM reporting.v_sensitivity_decision_coverage
    """).df()

    reference_actions = con.execute("""
        SELECT Scenario, Action
        FROM reporting.v_inventory_launch_readiness
    """).df()

finally:
    con.close()

print("\n===== REPORTING SOURCE INVENTORY =====")

for name, df in {
    "Operating cases": cases,
    "Intervention evaluations": evaluations,
    "Action robustness": robustness,
    "Decision coverage": coverage
}.items():
    print(f"{name}: {len(df)} rows")

print("\nEvaluation fields:", evaluations.columns.tolist())
print("\nRobustness fields:", robustness.columns.tolist())
print("\nCoverage fields:", coverage.columns.tolist())

# 2. Verify the expected source counts
source_checks = {
    "162 operating cases": len(cases) == 162,
    "567 intervention evaluations": len(evaluations) == 567,
    "7 action robustness records": len(robustness) == 7,
    "2 decision coverage records": len(coverage) == 2,
    "7 original action definitions": len(reference_actions) == 7
}

print("\n===== SOURCE COUNT VALIDATION =====")

for name, passed in source_checks.items():
    print(f"{name}: {'PASS' if passed else 'REVIEW'}")

# 3. Validate unique operating cases
case_keys = ["Scenario", "Case_ID"]
evaluation_keys = ["Scenario", "Case_ID", "Action"]

assert set(case_keys).issubset(cases.columns)
assert set(evaluation_keys).issubset(evaluations.columns)

unique_cases_pass = not cases.duplicated(case_keys).any()
unique_evaluations_pass = not evaluations.duplicated(
    evaluation_keys
).any()

cases_by_scenario = (
    cases.groupby("Scenario")
    .size()
    .reset_index(name="Operating_Cases")
)

print("\n===== OPERATING CASES BY SCENARIO =====")
display(cases_by_scenario)

scenario_case_pass = (
    len(cases_by_scenario) == 2
    and (cases_by_scenario["Operating_Cases"] == 81).all()
)

print("Unique case identifiers:",
      "PASS" if unique_cases_pass else "REVIEW")

print("81 cases per scenario:",
      "PASS" if scenario_case_pass else "REVIEW")

print("Unique intervention evaluations:",
      "PASS" if unique_evaluations_pass else "REVIEW")

# 4. Independently reconstruct the expected evaluation grid
# Every operating case should have every applicable action.

expected_grid = cases[case_keys].merge(
    reference_actions[["Scenario", "Action"]],
    on="Scenario",
    how="inner",
    validate="many_to_many"
)

def sorted_keys(df, columns):
    return (
        df[columns]
        .astype(str)
        .sort_values(columns)
        .reset_index(drop=True)
    )

grid_pass = False

try:
    pd.testing.assert_frame_equal(
        sorted_keys(expected_grid, evaluation_keys),
        sorted_keys(evaluations, evaluation_keys)
    )
    grid_pass = True

except AssertionError as error:
    print("Evaluation grid discrepancy:", str(error)[:400])

print("\n===== INTERVENTION COVERAGE =====")

evaluations_by_action = (
    evaluations.groupby(["Scenario", "Action"])
    .size()
    .reset_index(name="Cases_Tested")
)

display(evaluations_by_action)

print("All expected case-action combinations:",
      "PASS" if grid_pass else "REVIEW")

# 5. Derive READY outcomes from the evaluation records
# Prefer the source status; verify against the 98% target
# when a fill-rate field is available.

if "Readiness_Status" in evaluations.columns:
    evaluations["_Ready"] = (
        evaluations["Readiness_Status"]
        .astype(str)
        .str.strip()
        .str.upper()
        .eq("READY")
    )

elif "Fill_Rate_Pct" in evaluations.columns:
    evaluations["_Ready"] = (
        pd.to_numeric(
            evaluations["Fill_Rate_Pct"],
            errors="raise"
        ) >= 98
    )
    print("\nReadiness derived using >=98% fill rate.")

else:
    raise ValueError(
        "No readiness status or fill-rate field available. "
        "Inspect the evaluation columns."
    )

if (
    "Readiness_Status" in evaluations.columns
    and "Fill_Rate_Pct" in evaluations.columns
):
    calculated_ready = (
        pd.to_numeric(
            evaluations["Fill_Rate_Pct"],
            errors="raise"
        ) >= 98
    )

    readiness_rule_pass = (
        calculated_ready == evaluations["_Ready"]
    ).all()

    print("\n98% readiness threshold consistency:",
          "PASS" if readiness_rule_pass else "REVIEW")
else:
    readiness_rule_pass = True
    print("\n98% threshold cross-check: NOT AVAILABLE")

# 6. Independently recalculate action robustness
calculated_robustness = (
    evaluations.groupby(
        ["Scenario", "Action"], as_index=False
    )
    .agg(
        Cases_Tested=("Case_ID", "size"),
        Ready_Cases=("_Ready", "sum")
    )
)

calculated_robustness["Ready_Rate_Pct"] = (
    calculated_robustness["Ready_Cases"]
    / calculated_robustness["Cases_Tested"]
    * 100
)

print("\n===== RECALCULATED ACTION ROBUSTNESS =====")

display(calculated_robustness.round(2))

# Compare to the stored seven-row summary
robustness_metrics = [
    "Cases_Tested",
    "Ready_Cases",
    "Ready_Rate_Pct"
]

assert set(robustness_metrics).issubset(
    robustness.columns
)

robustness_comparison = calculated_robustness.merge(
    robustness[
        ["Scenario", "Action", *robustness_metrics]
    ],
    on=["Scenario", "Action"],
    how="outer",
    suffixes=("_Calculated", "_Reported"),
    indicator=True
)

robustness_pass = (
    len(robustness_comparison) == 7
    and (robustness_comparison["_merge"] == "both").all()
)

for metric in robustness_metrics:
    actual = pd.to_numeric(
        robustness_comparison[f"{metric}_Calculated"],
        errors="raise"
    )
    reported_value = pd.to_numeric(
        robustness_comparison[f"{metric}_Reported"],
        errors="raise"
    )

    robustness_pass = (
        robustness_pass
        and np.allclose(
            actual, reported_value,
            atol=0.02, rtol=0
        )
    )

print("\nAction robustness reconciliation:",
      "PASS" if robustness_pass else "REVIEW")

# 7. Independently calculate scenario-level decision coverage
# A case has a READY option if at least one applicable
# intervention achieves READY status.

case_readiness = (
    evaluations.groupby(case_keys, as_index=False)
    .agg(Has_Ready_Action=("_Ready", "any"))
)

calculated_coverage = (
    case_readiness.groupby("Scenario", as_index=False)
    .agg(
        Operating_Cases=("Case_ID", "size"),
        Cases_With_Ready_Action=("Has_Ready_Action", "sum")
    )
)

calculated_coverage["No_Ready_Option_Cases"] = (
    calculated_coverage["Operating_Cases"]
    - calculated_coverage["Cases_With_Ready_Action"]
)

calculated_coverage["Ready_Option_Coverage_Pct"] = (
    calculated_coverage["Cases_With_Ready_Action"]
    / calculated_coverage["Operating_Cases"]
    * 100
)

print("\n===== RECALCULATED DECISION COVERAGE =====")
display(calculated_coverage.round(2))

coverage_metrics = [
    "Operating_Cases",
    "Cases_With_Ready_Action",
    "No_Ready_Option_Cases",
    "Ready_Option_Coverage_Pct"
]

assert set(coverage_metrics).issubset(coverage.columns)

coverage_comparison = calculated_coverage.merge(
    coverage[["Scenario", *coverage_metrics]],
    on="Scenario",
    how="outer",
    suffixes=("_Calculated", "_Reported"),
    indicator=True
)

coverage_pass = (
    len(coverage_comparison) == 2
    and (coverage_comparison["_merge"] == "both").all()
)

for metric in coverage_metrics:
    actual = pd.to_numeric(
        coverage_comparison[f"{metric}_Calculated"],
        errors="raise"
    )
    reported_value = pd.to_numeric(
        coverage_comparison[f"{metric}_Reported"],
        errors="raise"
    )

    coverage_pass = (
        coverage_pass
        and np.allclose(
            actual, reported_value,
            atol=0.02, rtol=0
        )
    )

print("\nDecision coverage reconciliation:",
      "PASS" if coverage_pass else "REVIEW")

# 8. Final validation summary
checks = {
    **source_checks,
    "Unique operating cases": unique_cases_pass,
    "81 cases per scenario": scenario_case_pass,
    "Unique evaluations": unique_evaluations_pass,
    "Complete case-action grid": grid_pass,
    "Readiness threshold consistency": readiness_rule_pass,
    "Action robustness calculations": robustness_pass,
    "Decision coverage calculations": coverage_pass
}

print("\n===== STEP 8 VALIDATION SUMMARY =====")

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'REVIEW'}")

if all(checks.values()):
    print("\n===== STEP 8 COMPLETE: PASS =====")
    print("Sensitivity coverage and robustness reconciled.")
else:
    print("\n===== STEP 8 REQUIRES REVIEW =====")
    print("Investigate discrepancies before proceeding.")

print("Read-only validation: source data unchanged.")


===== SENSITIVITY ANALYSIS VALIDATION =====

===== REPORTING SOURCE INVENTORY =====
Operating cases: 162 rows
Intervention evaluations: 567 rows
Action robustness: 7 rows
Decision coverage: 2 rows

Evaluation fields: ['Case_ID', 'Scenario', 'Action', 'Demand_Adjustment', 'Starting_Inventory', 'Arrival_Week_Adjustment', 'Shortage_Penalty_USD', 'Effective_Original_Arrival_Week', 'Total_Demand', 'Total_Fulfilled', 'Total_Shortage', 'Shortage_Weeks', 'Fill_Rate_Pct', 'Readiness_Status', 'Received_Extra_Units', 'Outstanding_Units', 'Ending_Inventory', 'Shortage_Exposure_USD', 'Emergency_Cost_USD', 'Delivery_Cost_USD', 'Expedite_Cost_USD', 'Total_Exposure_USD', 'Baseline_Shortage', 'Baseline_Fill_Rate', 'Baseline_Exposure_USD', 'Baseline_Readiness', 'Shortage_Reduction_Units', 'Fill_Rate_Improvement_pp', 'Exposure_Reduction_USD', 'Is_Ready', 'Is_Improved_Exposure', 'Is_Shortage_Free']

Robustness fields: ['Scenario', 'Action', 'Cases_Tested', 'Ready_Cases', 'Shortage_Free_Cases', 'Average_Sh

,Scenario,Operating_Cases
0,Demand Surge,81
1,Supplier Delay,81


Unique case identifiers: PASS
81 cases per scenario: PASS
Unique intervention evaluations: PASS

===== INTERVENTION COVERAGE =====


,Scenario,Action,Cases_Tested
0,Demand Surge,Add 276 Units - Week 4,81
1,Demand Surge,Add 276 Units - Week 5,81
2,Demand Surge,No Intervention,81
3,Supplier Delay,Bridge 660 Units - Week 3,81
4,Supplier Delay,Bridge 660 Units - Week 4,81
5,Supplier Delay,Expedite Original 900 - Week 3,81
6,Supplier Delay,No Intervention,81


All expected case-action combinations: PASS

98% readiness threshold consistency: REVIEW

===== RECALCULATED ACTION ROBUSTNESS =====


,Scenario,Action,Cases_Tested,Ready_Cases,Ready_Rate_Pct
0,Demand Surge,Add 276 Units - Week 4,81,33,40.74
1,Demand Surge,Add 276 Units - Week 5,81,21,25.93
2,Demand Surge,No Intervention,81,21,25.93
3,Supplier Delay,Bridge 660 Units - Week 3,81,54,66.67
4,Supplier Delay,Bridge 660 Units - Week 4,81,27,33.33
5,Supplier Delay,Expedite Original 900 - Week 3,81,45,55.56
6,Supplier Delay,No Intervention,81,9,11.11



Action robustness reconciliation: PASS

===== RECALCULATED DECISION COVERAGE =====


,Scenario,Operating_Cases,Cases_With_Ready_Action,No_Ready_Option_Cases,Ready_Option_Coverage_Pct
0,Demand Surge,81,33,48,40.74
1,Supplier Delay,81,57,24,70.37



Decision coverage reconciliation: PASS

===== STEP 8 VALIDATION SUMMARY =====
162 operating cases: PASS
567 intervention evaluations: PASS
7 action robustness records: PASS
2 decision coverage records: PASS
7 original action definitions: PASS
Unique operating cases: PASS
81 cases per scenario: PASS
Unique evaluations: PASS
Complete case-action grid: PASS
Readiness threshold consistency: REVIEW
Action robustness calculations: PASS
Decision coverage calculations: PASS

===== STEP 8 REQUIRES REVIEW =====
Investigate discrepancies before proceeding.
Read-only validation: source data unchanged.


In [11]:
# STEP 8A: DIAGNOSE READINESS STATUS DISCREPANCIES
# Read-only analysis. No reporting data modified.

print("===== READINESS STATUS DIAGNOSTIC =====")

diagnostic = evaluations.copy()

diagnostic["Fill_Rate_Pct"] = pd.to_numeric(
    diagnostic["Fill_Rate_Pct"],
    errors="raise"
)

diagnostic["Status_Is_READY"] = (
    diagnostic["Readiness_Status"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("READY")
)

diagnostic["Meets_98pct_Threshold"] = (
    diagnostic["Fill_Rate_Pct"] >= 98
)

diagnostic["Readiness_Mismatch"] = (
    diagnostic["Status_Is_READY"]
    != diagnostic["Meets_98pct_Threshold"]
)

mismatches = diagnostic[
    diagnostic["Readiness_Mismatch"]
].copy()

print("\n===== MISMATCH SUMMARY =====")
print("Total evaluations:", len(diagnostic))
print("Disagreements:", len(mismatches))

print("\n===== STATUS VS THRESHOLD =====")

display(
    diagnostic.groupby(
        ["Readiness_Status", "Meets_98pct_Threshold"],
        dropna=False
    )
    .size()
    .reset_index(name="Evaluations")
)

print("\n===== MISMATCH DETAILS =====")

inspect_fields = [
    "Case_ID",
    "Scenario",
    "Action",
    "Fill_Rate_Pct",
    "Readiness_Status",
    "Total_Shortage",
    "Shortage_Weeks",
    "Outstanding_Units",
    "Ending_Inventory"
]

inspect_fields = [
    col for col in inspect_fields
    if col in mismatches.columns
]

display(
    mismatches[inspect_fields]
    .sort_values("Fill_Rate_Pct")
    .head(25)
    .round(3)
)

print("\n===== ALTERNATIVE READINESS CONDITIONS =====")

# Evaluate possible explanations without assuming
# that any one rule is the correct business rule.

candidate_rules = {
    "Fill rate >= 98%":
        diagnostic["Fill_Rate_Pct"] >= 98,
    "Fill rate = 100%":
        np.isclose(
            diagnostic["Fill_Rate_Pct"], 100,
            atol=1e-6
        )
}

if "Shortage_Weeks" in diagnostic.columns:
    candidate_rules["98% and zero shortage weeks"] = (
        (diagnostic["Fill_Rate_Pct"] >= 98)
        & (diagnostic["Shortage_Weeks"] == 0)
    )

if "Outstanding_Units" in diagnostic.columns:
    candidate_rules["98% and zero outstanding units"] = (
        (diagnostic["Fill_Rate_Pct"] >= 98)
        & (diagnostic["Outstanding_Units"] == 0)
    )

rule_results = []

for name, predicted_ready in candidate_rules.items():
    rule_results.append({
        "Candidate_Rule": name,
        "Disagreements": int(
            (
                predicted_ready
                != diagnostic["Status_Is_READY"]
            ).sum()
        )
    })

display(
    pd.DataFrame(rule_results)
    .sort_values("Disagreements")
)

print("\n===== STEP 8A COMPLETE =====")
print("Diagnostic only; no readiness rule changed.")


===== READINESS STATUS DIAGNOSTIC =====

===== MISMATCH SUMMARY =====
Total evaluations: 567
Disagreements: 6

===== STATUS VS THRESHOLD =====


,Readiness_Status,Meets_98pct_Threshold,Evaluations
0,AT RISK,False,273
1,READY,True,210
2,WATCH,False,78
3,WATCH,True,6



===== MISMATCH DETAILS =====


,Case_ID,Scenario,Action,Fill_Rate_Pct,Readiness_Status,Total_Shortage,Shortage_Weeks,Outstanding_Units,Ending_Inventory
544,CASE_157,Supplier Delay,Bridge 660 Units - Week 3,99.494,WATCH,12,1,900,0
548,CASE_158,Supplier Delay,Bridge 660 Units - Week 3,99.494,WATCH,12,1,900,0
552,CASE_159,Supplier Delay,Bridge 660 Units - Week 3,99.494,WATCH,12,1,900,0
556,CASE_160,Supplier Delay,Bridge 660 Units - Week 3,99.494,WATCH,12,1,900,0
560,CASE_161,Supplier Delay,Bridge 660 Units - Week 3,99.494,WATCH,12,1,900,0
564,CASE_162,Supplier Delay,Bridge 660 Units - Week 3,99.494,WATCH,12,1,900,0



===== ALTERNATIVE READINESS CONDITIONS =====


,Candidate_Rule,Disagreements
1,Fill rate = 100%,0
2,98% and zero shortage weeks,0
0,Fill rate >= 98%,6
3,98% and zero outstanding units,69



===== STEP 8A COMPLETE =====
Diagnostic only; no readiness rule changed.


In [12]:

# STEP 8B: VERIFY SHORTAGE-AWARE READINESS

import pandas as pd
import numpy as np

print("===== READINESS RULE VALIDATION =====")

# Work from the existing evaluation data
readiness_df = evaluations.copy()

readiness_df["Fill_Rate_Pct"] = pd.to_numeric(
    readiness_df["Fill_Rate_Pct"], errors="raise"
)

readiness_df["Shortage_Weeks"] = pd.to_numeric(
    readiness_df["Shortage_Weeks"], errors="raise"
)

# Original recorded READY classification
actual_ready = (
    readiness_df["Readiness_Status"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("READY")
)

# Test the shortage-aware business rule
calculated_ready = (
    (readiness_df["Fill_Rate_Pct"] >= 98)
    & (readiness_df["Shortage_Weeks"] == 0)
)

# Independently compare every evaluation
disagreements = (
    actual_ready != calculated_ready
).sum()

print("Total evaluations:", len(readiness_df))
print("Recorded READY cases:", int(actual_ready.sum()))
print("Calculated READY cases:", int(calculated_ready.sum()))
print("Classification disagreements:", disagreements)

rule_pass = (
    len(readiness_df) == 567
    and disagreements == 0
)

print("\n===== READINESS VALIDATION =====")
print(
    "Shortage-aware classification:",
    "PASS" if rule_pass else "REVIEW"
)

# Verify there are no READY cases with shortages
ready_with_shortages = (
    actual_ready
    & (readiness_df["Shortage_Weeks"] > 0)
).sum()

print(
    "READY cases with shortages:",
    int(ready_with_shortages)
)

# Update the prior audit's threshold check
# Only if the shortage-aware rule matches all records
revised_checks = dict(checks)

revised_checks["Readiness threshold consistency"] = (
    rule_pass
)

print("\n===== STEP 8 REVISED VALIDATION =====")

for name, passed in revised_checks.items():
    print(
        f"{name}:",
        "PASS" if passed else "REVIEW"
    )

if all(revised_checks.values()):
    print("\n===== STEP 8B: EMPIRICAL CHECKS PASSED =====")
    print("All 567 classifications reconciled.")
    print("Confirm the rule in the original simulation code.")
else:
    print("\n===== STEP 8B REQUIRES REVIEW =====")

print("Source files unchanged.")


===== READINESS RULE VALIDATION =====
Total evaluations: 567
Recorded READY cases: 210
Calculated READY cases: 210
Classification disagreements: 0

===== READINESS VALIDATION =====
Shortage-aware classification: PASS
READY cases with shortages: 0

===== STEP 8 REVISED VALIDATION =====
162 operating cases: PASS
567 intervention evaluations: PASS
7 action robustness records: PASS
2 decision coverage records: PASS
7 original action definitions: PASS
Unique operating cases: PASS
81 cases per scenario: PASS
Unique evaluations: PASS
Complete case-action grid: PASS
Readiness threshold consistency: PASS
Action robustness calculations: PASS
Decision coverage calculations: PASS

===== STEP 8B: EMPIRICAL CHECKS PASSED =====
All 567 classifications reconciled.
Confirm the rule in the original simulation code.
Source files unchanged.


In [13]:

# STEP 8C: LOCATE ORIGINAL READINESS LOGIC
# Read-only search of the original project notebooks

import json
from pathlib import Path

print("===== ORIGINAL READINESS LOGIC AUDIT =====")

source_notebooks = sorted(
    p for p in PROJECT_ROOT.glob("*.ipynb")
    if not p.name.startswith("13_")
)

matches = []

for notebook_path in source_notebooks:
    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    for cell_number, cell in enumerate(
        notebook.get("cells", []), start=1
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lower = source.lower()

        if "readiness_status" not in lower:
            continue

        # Prioritize code containing a status rule
        score = 0
        if "case" in lower and "when" in lower:
            score += 5
        if "shortage_weeks" in lower:
            score += 5
        if "fill_rate_pct" in lower:
            score += 3
        if "'ready'" in lower or '"ready"' in lower:
            score += 3

        matches.append((
            score, notebook_path.name,
            cell_number, source
        ))

matches.sort(reverse=True)

print("Matching code cells:", len(matches))

for score, filename, cell_number, source in matches[:8]:
    print(f"\n===== {filename} | CELL {cell_number} =====")

    lines = source.splitlines()

    matching_lines = [
        i for i, line in enumerate(lines)
        if any(term in line.lower() for term in [
            "readiness_status",
            "shortage_weeks",
            "fill_rate_pct",
            "'ready'",
            '"ready"'
        ])
    ]

    shown = set()

    for i in matching_lines[:5]:
        start = max(0, i - 4)
        end = min(len(lines), i + 5)

        for j in range(start, end):
            if j not in shown:
                print(f"{j+1:>4}: {lines[j][:180]}")
                shown.add(j)

        print("...")

print("\n===== STEP 8C SEARCH COMPLETE =====")
print("Review the original CASE/IF conditions.")
print("No source files modified.")


===== ORIGINAL READINESS LOGIC AUDIT =====
Matching code cells: 7

===== 09_inventory_sensitivity_analysis.ipynb | CELL 2 =====
 189:     total_shortage = int(
 190:         weekly_df["Shortage_Units"].sum()
 191:     )
 192: 
 193:     shortage_weeks = int(
 194:         (weekly_df["Shortage_Units"] > 0).sum()
 195:     )
 196: 
 197:     fill_rate = (
...
 201:     )
 202: 
 203:     if (
 204:         fill_rate >= launch_fill_target
 205:         and shortage_weeks == 0
 206:     ):
 207: 
 208:         readiness = "READY"
 209: 
...
 210:     elif fill_rate >= watch_threshold:
 211: 
 212:         readiness = "WATCH"
...
 293:             original_arrival_week,
 294:         "Total_Demand": total_demand,
 295:         "Total_Fulfilled": total_fulfilled,
 296:         "Total_Shortage": total_shortage,
 297:         "Shortage_Weeks": shortage_weeks,
 298:         "Fill_Rate_Pct": fill_rate,
 299:         "Readiness_Status": readiness,
 300:         "Received_Extra_Units": received_ex

In [14]:
# STEP 9: DUCKDB WAREHOUSE INTEGRITY VALIDATION
# Read-only catalog and reporting-view audit

import pandas as pd
import duckdb

print("===== WAREHOUSE INTEGRITY VALIDATION =====")

warehouse_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert warehouse_path.exists(), "Warehouse not found."

con = duckdb.connect(
    str(warehouse_path),
    read_only=True
)

try:
    # 1. Inspect all permanent warehouse objects
    catalog = con.execute("""
        SELECT
            table_schema,
            table_name,
            table_type
        FROM information_schema.tables
        WHERE table_schema NOT IN (
            'information_schema',
            'pg_catalog'
        )
        ORDER BY table_schema, table_type, table_name
    """).df()

    print("\n===== WAREHOUSE OBJECT INVENTORY =====")

    display(
        catalog.groupby(
            ["table_schema", "table_type"]
        )
        .size()
        .reset_index(name="Object_Count")
    )

    # 2. Identify reporting views
    reporting_views = catalog[
        (catalog["table_schema"] == "reporting")
        & (catalog["table_type"] == "VIEW")
    ].copy()

    print("\nReporting views found:", len(reporting_views))

    display(
        reporting_views[
            ["table_name", "table_type"]
        ].reset_index(drop=True)
    )

    # 3. Query-test every reporting view
    view_health = []

    for name in reporting_views["table_name"]:

        # Escape identifiers safely
        quoted_name = '"' + name.replace('"', '""') + '"'

        try:
            con.execute(
                f'SELECT * FROM "reporting".{quoted_name} LIMIT 1'
            ).fetchall()

            view_health.append({
                "View": name,
                "Status": "PASS",
                "Error": ""
            })

        except Exception as error:
            view_health.append({
                "View": name,
                "Status": "REVIEW",
                "Error": str(error)[:200]
            })

    view_health_df = pd.DataFrame(
        view_health,
        columns=["View", "Status", "Error"]
    )

    print("\n===== REPORTING VIEW HEALTH =====")
    display(view_health_df)

    # 4. Check critical reporting outputs
    expected_records = {
        "v_airpods_weekly_demand": 51,
        "v_forecast_model_performance": 8,
        "v_external_signal_model_comparison": 3,
        "v_external_signal_aligned_demand": 51,
        "v_inventory_weekly": 12,
        "v_inventory_scenario_performance": 3,
        "v_inventory_launch_readiness": 7,
        "v_inventory_candidate_actions": 2,
        "v_sensitivity_decision_cases": 162,
        "v_sensitivity_intervention_evaluations": 567,
        "v_sensitivity_action_robustness": 7,
        "v_sensitivity_decision_coverage": 2
    }

    available = set(reporting_views["table_name"])
    record_checks = []

    for name, expected in expected_records.items():

        if name not in available:
            record_checks.append({
                "Reporting_View": name,
                "Expected_Rows": expected,
                "Actual_Rows": None,
                "Status": "MISSING"
            })
            continue

        quoted_name = '"' + name.replace('"', '""') + '"'

        try:
            actual = con.execute(
                f'SELECT COUNT(*) FROM "reporting".{quoted_name}'
            ).fetchone()[0]

            record_checks.append({
                "Reporting_View": name,
                "Expected_Rows": expected,
                "Actual_Rows": actual,
                "Status": (
                    "PASS" if actual == expected else "REVIEW"
                )
            })

        except Exception as error:
            record_checks.append({
                "Reporting_View": name,
                "Expected_Rows": expected,
                "Actual_Rows": None,
                "Status": f"ERROR: {str(error)[:80]}"
            })

    record_checks_df = pd.DataFrame(record_checks)

    print("\n===== CRITICAL REPORTING RECORD COUNTS =====")
    display(record_checks_df)

finally:
    con.close()

# 5. Summarize results
view_health_pass = (
    len(view_health_df) > 0
    and view_health_df["Status"].eq("PASS").all()
)

record_counts_pass = (
    record_checks_df["Status"].eq("PASS").all()
)

catalog_pass = len(reporting_views) > 0

print("\n===== STEP 9 VALIDATION SUMMARY =====")

checks_step9 = {
    "Reporting catalog available": catalog_pass,
    "All reporting views query successfully": view_health_pass,
    "Critical reporting counts match": record_counts_pass
}

for label, passed in checks_step9.items():
    print(
        f"{label}:",
        "PASS" if passed else "REVIEW"
    )

if all(checks_step9.values()):
    print("\n===== STEP 9 COMPLETE: PASS =====")
    print("Reporting warehouse integrity verified.")
else:
    print("\n===== STEP 9 REQUIRES REVIEW =====")

print("Read-only audit: source warehouse unchanged.")


===== WAREHOUSE INTEGRITY VALIDATION =====

===== WAREHOUSE OBJECT INVENTORY =====


,table_schema,table_type,Object_Count
0,demand,BASE TABLE,8
1,ecommerce,BASE TABLE,4
2,intent,BASE TABLE,2
3,inventory,BASE TABLE,8
4,metadata,BASE TABLE,3
5,reporting,VIEW,18
6,retail,BASE TABLE,7
7,sensitivity,BASE TABLE,8
8,signals,BASE TABLE,6



Reporting views found: 18


,table_name,table_type
0,v_airpods_weekly_demand,VIEW
1,v_external_signal_aligned_demand,VIEW
2,v_external_signal_model_comparison,VIEW
3,v_forecast_lead_time_baseline,VIEW
4,v_forecast_model_performance,VIEW
5,v_forecast_multihorizon_performance,VIEW
6,v_headphone_daily_signals,VIEW
7,v_inventory_candidate_actions,VIEW
8,v_inventory_launch_readiness,VIEW
9,v_inventory_scenario_performance,VIEW



===== REPORTING VIEW HEALTH =====


,View,Status,Error
0,v_airpods_weekly_demand,PASS,
1,v_external_signal_aligned_demand,PASS,
2,v_external_signal_model_comparison,PASS,
3,v_forecast_lead_time_baseline,PASS,
4,v_forecast_model_performance,PASS,
5,v_forecast_multihorizon_performance,PASS,
6,v_headphone_daily_signals,PASS,
7,v_inventory_candidate_actions,PASS,
8,v_inventory_launch_readiness,PASS,
9,v_inventory_scenario_performance,PASS,



===== CRITICAL REPORTING RECORD COUNTS =====


,Reporting_View,Expected_Rows,Actual_Rows,Status
0,v_airpods_weekly_demand,51,51,PASS
1,v_forecast_model_performance,8,8,PASS
2,v_external_signal_model_comparison,3,3,PASS
3,v_external_signal_aligned_demand,51,51,PASS
4,v_inventory_weekly,12,12,PASS
5,v_inventory_scenario_performance,3,3,PASS
6,v_inventory_launch_readiness,7,7,PASS
7,v_inventory_candidate_actions,2,2,PASS
8,v_sensitivity_decision_cases,162,162,PASS
9,v_sensitivity_intervention_evaluations,567,567,PASS



===== STEP 9 VALIDATION SUMMARY =====
Reporting catalog available: PASS
All reporting views query successfully: PASS
Critical reporting counts match: PASS

===== STEP 9 COMPLETE: PASS =====
Reporting warehouse integrity verified.
Read-only audit: source warehouse unchanged.


In [ ]:

# STEP 10: INDEPENDENT WAREHOUSE SNAPSHOT RECONCILIATION

# Compares existing source and rebuilt DuckDB reporting views
# READ ONLY — does not modify either database

import pandas as pd
import numpy as np
import duckdb
from datetime import date, datetime

print("===== WAREHOUSE SNAPSHOT RECONCILIATION =====")

source_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_candidate.duckdb"
)

rebuild_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert source_path.exists(), f"Source DB missing: {source_path}"
assert rebuild_path.exists(), f"Rebuilt DB missing: {rebuild_path}"
assert source_path.resolve() != rebuild_path.resolve()

print("\n===== DATABASE FILES =====")
print("Source:", source_path.name)
print("Rebuild:", rebuild_path.name)

# 1. Open both databases in read-only mode
source_con = duckdb.connect(str(source_path), read_only=True)
rebuild_con = duckdb.connect(str(rebuild_path), read_only=True)

try:
    # 2. Identify reporting views in each database
    view_query = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'reporting'
          AND table_type = 'VIEW'
        ORDER BY table_name
    """

    source_views = set(
        source_con.execute(view_query).df()["table_name"]
    )

    rebuild_views = set(
        rebuild_con.execute(view_query).df()["table_name"]
    )

    print("\n===== REPORTING SCHEMA COMPARISON =====")
    print("Source views:", len(source_views))
    print("Rebuilt views:", len(rebuild_views))
    print("Shared views:", len(source_views & rebuild_views))

    missing_from_rebuild = sorted(
        source_views - rebuild_views
    )

    extra_in_rebuild = sorted(
        rebuild_views - source_views
    )

    print("Missing from rebuild:", missing_from_rebuild)
    print("Extra in rebuild:", extra_in_rebuild)

    schema_match = source_views == rebuild_views

    print(
        "Reporting view inventory:",
        "PASS" if schema_match else "REVIEW"
    )

    # 3. Standardize values for order-independent comparisons
    def normalize_value(value):
        if pd.isna(value):
            return "<NULL>"

        if isinstance(value, (bool, np.bool_)):
            return f"BOOL:{bool(value)}"

        if isinstance(value, (int, float, np.number)):
            return f"NUM:{float(value):.8f}"

        if isinstance(value, (date, datetime, pd.Timestamp)):
            return f"DATE:{pd.Timestamp(value).isoformat()}"

        return f"TEXT:{str(value)}"

    def normalize_rows(df):
        ordered_columns = sorted(df.columns)

        records = [
            tuple(normalize_value(value) for value in row)
            for row in df[ordered_columns].itertuples(
                index=False, name=None
            )
        ]

        return sorted(records)

    # 4. Independently compare every shared reporting view
    comparison_results = []

    for view in sorted(source_views & rebuild_views):
        quoted = '"' + view.replace('"', '""') + '"'
        query = f'SELECT * FROM "reporting".{quoted}'

        try:
            source_df = source_con.execute(query).df()
            rebuild_df = rebuild_con.execute(query).df()

            columns_match = (
                set(source_df.columns)
                == set(rebuild_df.columns)
            )

            counts_match = (
                len(source_df) == len(rebuild_df)
            )

            content_match = False

            if columns_match and counts_match:
                content_match = (
                    normalize_rows(source_df)
                    == normalize_rows(rebuild_df)
                )

            passed = (
                columns_match
                and counts_match
                and content_match
            )

            comparison_results.append({
                "Reporting_View": view,
                "Source_Rows": len(source_df),
                "Rebuild_Rows": len(rebuild_df),
                "Columns_Match": columns_match,
                "Content_Match": content_match,
                "Status": "PASS" if passed else "REVIEW"
            })

        except Exception as error:
            comparison_results.append({
                "Reporting_View": view,
                "Source_Rows": None,
                "Rebuild_Rows": None,
                "Columns_Match": False,
                "Content_Match": False,
                "Status": "ERROR"
            })
            print(f"Error in {view}: {str(error)[:200]}")

finally:
    source_con.close()
    rebuild_con.close()

# 5. Display full reconciliation
results = pd.DataFrame(comparison_results)

print("\n===== VIEW-BY-VIEW RECONCILIATION =====")
display(results)

all_views_match = (
    len(results) > 0
    and results["Status"].eq("PASS").all()
)

full_parity = schema_match and all_views_match

print("\n===== STEP 10 VALIDATION SUMMARY =====")

print(
    "Reporting schemas match:",
    "PASS" if schema_match else "REVIEW"
)

print(
    "Shared reporting view contents match:",
    "PASS" if all_views_match else "REVIEW"
)

print(
    "Overall warehouse snapshot parity:",
    "PASS" if full_parity else "REVIEW"
)

if full_parity:
    print("\n===== STEP 10 COMPLETE: PASS =====")
    print("Existing source and rebuilt reporting data match.")
else:
    print("\n===== STEP 10 REQUIRES REVIEW =====")
    print("Inspect differences before claiming rebuild parity.")

print("Both database files remain unchanged.")


===== WAREHOUSE SNAPSHOT RECONCILIATION =====

===== DATABASE FILES =====
Source: demand_intelligence_signals_candidate.duckdb
Rebuild: demand_intelligence_signals_manifest_rebuild.duckdb

===== REPORTING SCHEMA COMPARISON =====
Source views: 18
Rebuilt views: 18
Shared views: 18
Missing from rebuild: []
Extra in rebuild: []
Reporting view inventory: PASS

===== VIEW-BY-VIEW RECONCILIATION =====


,Reporting_View,Source_Rows,Rebuild_Rows,Columns_Match,Content_Match,Status
0,v_airpods_weekly_demand,51,51,True,True,PASS
1,v_external_signal_aligned_demand,51,51,True,True,PASS
2,v_external_signal_model_comparison,3,3,True,True,PASS
3,v_forecast_lead_time_baseline,4,4,True,True,PASS
4,v_forecast_model_performance,8,8,True,True,PASS
5,v_forecast_multihorizon_performance,15,15,True,True,PASS
6,v_headphone_daily_signals,61,61,True,True,PASS
7,v_inventory_candidate_actions,2,2,True,True,PASS
8,v_inventory_launch_readiness,7,7,True,True,PASS
9,v_inventory_scenario_performance,3,3,True,True,PASS



===== STEP 10 VALIDATION SUMMARY =====
Reporting schemas match: PASS
Shared reporting view contents match: PASS
Overall warehouse snapshot parity: PASS

===== STEP 10 COMPLETE: PASS =====
Existing source and rebuilt reporting data match.
Both database files remain unchanged.


In [16]:
# STEP 11: REPRODUCIBILITY READINESS AUDIT
# Read-only inspection of rebuild instructions and environment

import json
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("===== REPRODUCIBILITY READINESS AUDIT =====")

# 1. Locate the original warehouse rebuild notebook

rebuild_notebooks = sorted(
    PROJECT_ROOT.glob("10_*.ipynb")
)

print("\n===== REBUILD NOTEBOOK =====")

if rebuild_notebooks:
    for path in rebuild_notebooks:
        print("Found:", path.name)

    rebuild_notebook = rebuild_notebooks[0]

    notebook_data = json.loads(
        rebuild_notebook.read_text(encoding="utf-8")
    )

    code_cells = [
        "".join(cell.get("source", []))
        for cell in notebook_data.get("cells", [])
        if cell.get("cell_type") == "code"
    ]

    print("Code cells:", len(code_cells))

    print("\n===== REBUILD OPERATIONS FOUND =====")

    full_code = "\n".join(code_cells).lower()

    operations = {
        "Manifest handling": "manifest",
        "DuckDB connection": "duckdb.connect",
        "SQL view creation": "create view",
        "SQL table creation": "create table",
        "Source CSV loading": "read_csv",
        "Warehouse rebuilding": "rebuild",
    }

    for label, keyword in operations.items():
        found = keyword in full_code
        print(
            f"{label}:",
            "FOUND" if found else "NOT DETECTED"
        )

    print("\n===== RELEVANT REBUILD CODE CELLS =====")

    shown = 0

    for index, source in enumerate(code_cells, start=1):
        if any(term in source.lower() for term in [
            "manifest",
            "rebuild",
            "duckdb.connect"
        ]):
            print(f"\nCode cell {index}:")
            print(source[:850])
            print("...")
            shown += 1

            if shown >= 4:
                break

else:
    print("REVIEW: Notebook 10 not found.")

# 2. Inspect configuration and reproducibility assets

print("\n===== CONFIGURATION FILES =====")

config_files = sorted(
    path for path in CONFIG_DIR.rglob("*")
    if path.is_file()
)

for path in config_files[:35]:
    print(path.relative_to(PROJECT_ROOT))

print("Total configuration files:", len(config_files))

# 3. Check environment documentation

print("\n===== ENVIRONMENT DOCUMENTATION =====")

environment_files = [
    "requirements.txt",
    "pyproject.toml",
    "environment.yml",
    "uv.lock",
    "README.md"
]

for filename in environment_files:
    path = PROJECT_ROOT / filename
    print(
        filename,
        "FOUND" if path.exists() else "NOT PRESENT"
    )

# 4. Record installed dependency versions

print("\n===== CURRENT PYTHON ENVIRONMENT =====")
print("Python:", sys.version.split()[0])

packages = [
    "pandas",
    "duckdb",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "statsmodels"
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# 5. Confirm existing data availability

print("\n===== INPUT / OUTPUT DIRECTORY INVENTORY =====")

directories = {
    "Processed data": PROCESSED_DIR,
    "Dashboard data": DASHBOARD_DIR,
    "Warehouse": WAREHOUSE_DIR,
    "Configuration": CONFIG_DIR
}

for name, directory in directories.items():
    files = (
        sum(p.is_file() for p in directory.rglob("*"))
        if directory.exists() else 0
    )

    print(f"{name}: {files} files")

print("\n===== STEP 11 INSPECTION COMPLETE =====")
print("Rebuild instructions and dependencies inventoried.")
print("No notebooks, databases or datasets modified.")
print("Fresh rebuild has NOT been executed.")


===== REPRODUCIBILITY READINESS AUDIT =====

===== REBUILD NOTEBOOK =====
Found: 10_reproducible_data_pipeline.ipynb
Code cells: 6

===== REBUILD OPERATIONS FOUND =====
Manifest handling: FOUND
DuckDB connection: FOUND
SQL view creation: FOUND
SQL table creation: FOUND
Source CSV loading: FOUND
Warehouse rebuilding: FOUND

===== RELEVANT REBUILD CODE CELLS =====

Code cell 1:
# STEP 1: AUDIT EXISTING DATA AND DATABASE ARCHITECTURE

from pathlib import Path
import pandas as pd
import duckdb

# ------------------------------------------------------------
# PART 1: LOCATE PROJECT ROOT
# ------------------------------------------------------------

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (
            (path / "data" / "processed").is_dir()
            and (
                path / "data" / "warehouse"
                / "demand_intelligence.duckdb"
            ).is_file()
        )
    ),
    None
)

assert project_root is not 

In [17]:
# STEP 12: INSPECT WAREHOUSE REBUILD DESTINATIONS

# READ ONLY — does not execute Notebook 10

import json
from pathlib import Path

print("===== REBUILD SAFETY INSPECTION =====")

notebook_path = (
    PROJECT_ROOT / "10_reproducible_data_pipeline.ipynb"
)

assert notebook_path.exists()

nb = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

keywords = [
    "duckdb.connect",
    ".duckdb",
    "db_path",
    "output_path",
    "rebuild_path",
    "candidate_path",
    "manifest",
    "unlink(",
    "os.remove",
    "shutil.",
    "mkdir(",
    "write_text(",
    "to_csv(",
    "read_csv",
    "create or replace",
    "drop table",
    "drop view"
]

print("\n===== REBUILD CODE REFERENCES =====")

total_matches = 0

for cell_num, cell in enumerate(nb.get("cells", []), 1):

    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()

    matches = [
        (line_num, line.strip())
        for line_num, line in enumerate(lines, 1)
        if any(
            keyword in line.lower()
            for keyword in keywords
        )
    ]

    if matches:
        print(f"\n--- Notebook cell {cell_num} ---")

        for line_num, line in matches[:20]:
            print(f"Line {line_num}: {line[:180]}")

        total_matches += len(matches)

print("\nTotal matching references:", total_matches)

print("\n===== EXISTING WAREHOUSE FILES =====")

for db in sorted(WAREHOUSE_DIR.glob("*.duckdb")):
    print(f"{db.name} | {db.stat().st_size / 1024**2:.2f} MB")

print("\n===== ENVIRONMENT DOCUMENTATION =====")

for filename in [
    "requirements.txt",
    "pyproject.toml",
    "environment.yml",
    "uv.lock",
    "README.md"
]:
    path = PROJECT_ROOT / filename

    print(
        filename,
        "FOUND" if path.exists() else "NOT PRESENT"
    )

print("\n===== STEP 12 COMPLETE =====")
print("Rebuild operations inspected.")
print("No code executed from Notebook 10.")
print("No databases or source files modified.")


===== REBUILD SAFETY INSPECTION =====

===== REBUILD CODE REFERENCES =====

--- Notebook cell 1 ---
Line 19: / "demand_intelligence.duckdb"
Line 36: / "demand_intelligence.duckdb"
Line 70: con = duckdb.connect(
Line 248: csv_df = pd.read_csv(file_path)

--- Notebook cell 2 ---
Line 12: config_dir.mkdir(parents=True, exist_ok=True)
Line 14: rebuild_path = (
Line 18: / "demand_intelligence_rebuild_candidate.duckdb"
Line 21: assert rebuild_path.resolve() != database_path.resolve()
Line 25: print("Rebuild candidate:", rebuild_path)
Line 33: # PART 2: CREATE SOURCE TABLE MANIFEST
Line 36: table_manifest = (
Line 45: assert len(table_manifest) == 37
Line 46: assert table_manifest["File_Name"].is_unique
Line 48: manifest_path = config_dir / "table_manifest.csv"
Line 50: table_manifest.to_csv(
Line 51: manifest_path,
Line 55: print("\n===== TABLE MANIFEST =====")
Line 56: print("Source tables documented:", len(table_manifest))
Line 59: table_manifest.groupby("Schema")
Line 69: reference_con = 

In [18]:
# STEP 13: FINAL REBUILD SAFETY CHECK
# Read-only inspection of Notebook 10

import json
import re

print("===== FINAL REBUILD SAFETY CHECK =====")

notebook_path = (
    PROJECT_ROOT / "10_reproducible_data_pipeline.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

patterns = [
    r"database_path",
    r"rebuild_path",
    r"duckdb\.connect",
    r"\.unlink\s*\(",
    r"os\.remove",
    r"shutil\.",
    r"\.mkdir\s*\(",
    r"\.write_text\s*\(",
    r"\.write_bytes\s*\(",
    r"\.to_csv\s*\(",
    r"\bCOPY\s+",
    r"\bopen\s*\(",
    r"manifest_path",
    r"output_path",
    r"candidate_path"
]

expression = re.compile(
    "|".join(patterns),
    flags=re.IGNORECASE
)

match_count = 0

for cell_number, cell in enumerate(
    nb.get("cells", []), start=1
):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(
        cell.get("source", [])
    ).splitlines()

    hit_indices = [
        i for i, line in enumerate(lines)
        if expression.search(line)
    ]

    if not hit_indices:
        continue

    print(f"\n===== NOTEBOOK CELL {cell_number} =====")

    # Show context around every relevant operation
    selected = set()

    for i in hit_indices:
        for j in range(
            max(0, i - 3),
            min(len(lines), i + 5)
        ):
            selected.add(j)

    previous = -2

    for i in sorted(selected):
        if i > previous + 1:
            print("    ...")

        print(f"{i + 1:>4}: {lines[i][:180]}")
        previous = i

    match_count += len(hit_indices)

print("\n===== INSPECTION SUMMARY =====")
print("Potentially relevant references:", match_count)

print("\n===== STEP 13 COMPLETE =====")
print("All relevant rebuild operations inspected.")
print("No notebook code executed.")
print("No database files changed.")


===== FINAL REBUILD SAFETY CHECK =====

===== NOTEBOOK CELL 1 =====
    ...
  29: 
  30: processed_dir = project_root / "data" / "processed"
  31: 
  32: database_path = (
  33:     project_root
  34:     / "data"
  35:     / "warehouse"
  36:     / "demand_intelligence.duckdb"
    ...
  40: print(project_root)
  41: 
  42: print("\n===== DATABASE LOCATION =====")
  43: print(database_path)
  44: 
  45: 
  46: # ------------------------------------------------------------
  47: # PART 2: INVENTORY PROCESSED CSV FILES
    ...
  67: # PART 3: OPEN EXISTING DATABASE IN READ-ONLY MODE
  68: # ------------------------------------------------------------
  69: 
  70: con = duckdb.connect(
  71:     str(database_path),
  72:     read_only=True
  73: )
  74: 
  75: try:

===== NOTEBOOK CELL 2 =====
    ...
   9: # ------------------------------------------------------------
  10: 
  11: config_dir = project_root / "config"
  12: config_dir.mkdir(parents=True, exist_ok=True)
  13: 
  14: rebuil

In [ ]:
# STEP 14: PREPARE ISOLATED REBUILD WORKSPACE
# Copies required inputs into a temporary project directory.
# Does NOT execute Notebook 10 or modify original files.

from pathlib import Path
import tempfile
import shutil
import json

print("===== ISOLATED REBUILD PREPARATION =====")

# 1. Create a separate temporary workspace
ISOLATED_ROOT = Path(
    tempfile.mkdtemp(
        prefix="hardware_demand_rebuild_"
    )
).resolve()

print("\n===== ISOLATED PROJECT LOCATION =====")
print(ISOLATED_ROOT)

# 2. Prepare the same directory structure
isolated_data = ISOLATED_ROOT / "data"
isolated_processed = isolated_data / "processed"
isolated_warehouse = isolated_data / "warehouse"

isolated_processed.mkdir(parents=True)
isolated_warehouse.mkdir(parents=True)

# 3. Copy processed source data
source_csvs = sorted(
    PROCESSED_DIR.glob("*.csv")
)

assert len(source_csvs) > 0

for source in source_csvs:
    shutil.copy2(
        source,
        isolated_processed / source.name
    )

print("\n===== SOURCE DATA COPIED =====")
print("Processed CSVs:", len(source_csvs))

# 4. Copy configuration files
if CONFIG_DIR.exists():
    shutil.copytree(
        CONFIG_DIR,
        ISOLATED_ROOT / "config",
        dirs_exist_ok=True
    )

print("Configuration copied:", CONFIG_DIR.exists())

# 5. Copy original source database only
source_database = (
    WAREHOUSE_DIR / "demand_intelligence.duckdb"
)

assert source_database.exists(), (
    "Original warehouse missing; review before rebuilding."
)

shutil.copy2(
    source_database,
    isolated_warehouse / source_database.name
)

print("Source database copied:", source_database.name)

# 6. Copy Notebook 10
original_notebook = (
    PROJECT_ROOT
    / "10_reproducible_data_pipeline.ipynb"
)

assert original_notebook.exists()

isolated_notebook = (
    ISOLATED_ROOT
    / original_notebook.name
)

shutil.copy2(
    original_notebook,
    isolated_notebook
)

print("Rebuild notebook copied:", isolated_notebook.name)

# 7. Check for literal references to the original root
nb_data = json.loads(
    isolated_notebook.read_text(encoding="utf-8")
)

notebook_code = "\n".join(
    "".join(cell.get("source", []))
    for cell in nb_data.get("cells", [])
    if cell.get("cell_type") == "code"
)

hardcoded_root = str(PROJECT_ROOT) in notebook_code

print("\n===== SAFETY CHECKS =====")
print(
    "Isolated root differs from original:",
    ISOLATED_ROOT != PROJECT_ROOT
)

print(
    "Hardcoded original project root:",
    "REVIEW" if hardcoded_root else "NOT FOUND"
)

print(
    "Rebuild candidate already exists:",
    (
        isolated_warehouse
        / "demand_intelligence_rebuild_candidate.duckdb"
    ).exists()
)

# 8. Confirm isolated workspace contents
print("\n===== ISOLATED WORKSPACE INVENTORY =====")

for name, directory in {
    "Processed inputs": isolated_processed,
    "Warehouse files": isolated_warehouse,
    "Configuration": ISOLATED_ROOT / "config"
}.items():

    count = (
        sum(p.is_file() for p in directory.rglob("*"))
        if directory.exists() else 0
    )

    print(f"{name}: {count} files")

# 9. Final result
ready = (
    ISOLATED_ROOT != PROJECT_ROOT
    and not hardcoded_root
    and len(source_csvs) > 0
    and isolated_notebook.exists()
    and (
        isolated_warehouse
        / "demand_intelligence.duckdb"
    ).exists()
)

print("\n===== STEP 14 RESULT =====")

if ready:
    print("ISOLATED WORKSPACE: PREPARED")
    print("Original project inputs remain unchanged.")
    print("Fresh rebuild has NOT yet been executed.")
else:
    print("REVIEW REQUIRED BEFORE EXECUTING REBUILD")

print("\n===== STEP 14 COMPLETE =====")


===== ISOLATED REBUILD PREPARATION =====

===== ISOLATED PROJECT LOCATION =====
/private/var/folders/9c/495l1cs96zj_67bnbpkg5lm80000gn/T/hardware_demand_rebuild_4zsp3ig5

===== SOURCE DATA COPIED =====
Processed CSVs: 43
Configuration copied: True
Source database copied: demand_intelligence.duckdb
Rebuild notebook copied: 10_reproducible_data_pipeline.ipynb

===== SAFETY CHECKS =====
Isolated root differs from original: True
Hardcoded original project root: NOT FOUND
Rebuild candidate already exists: False

===== ISOLATED WORKSPACE INVENTORY =====
Processed inputs: 43 files
Warehouse files: 1 files
Configuration: 14 files

===== STEP 14 RESULT =====
ISOLATED WORKSPACE: PREPARED
Original project inputs remain unchanged.
Fresh rebuild has NOT yet been executed.

===== STEP 14 COMPLETE =====


In [20]:
# STEP 15: EXECUTE ISOLATED WAREHOUSE REBUILD

# Runs a COPY of Notebook 10 in a separate Python process.
# Does not execute or overwrite the original notebook.

import subprocess
import sys
import json
import hashlib
from pathlib import Path

print("===== ISOLATED WAREHOUSE REBUILD =====")

# 1. Verify isolation
assert ISOLATED_ROOT.exists()
assert ISOLATED_ROOT.resolve() != PROJECT_ROOT.resolve()
assert isolated_notebook.exists()

assert isolated_notebook.resolve().is_relative_to(
    ISOLATED_ROOT.resolve()
)

print("\n===== EXECUTION ENVIRONMENT =====")
print("Python:", sys.executable)
print("Workspace:", ISOLATED_ROOT)
print("Notebook:", isolated_notebook.name)

# 2. Check the copied notebook for unsafe path changes
notebook_data = json.loads(
    isolated_notebook.read_text(encoding="utf-8")
)

code_cells = [
    "".join(cell.get("source", []))
    for cell in notebook_data["cells"]
    if cell.get("cell_type") == "code"
]

all_code = "\n".join(code_cells)

unsafe_patterns = [
    "os.chdir(",
    "Path.home(",
    ".expanduser(",
    "%cd ",
    "%%bash",
    "!rm ",
    "!mv "
]

found_unsafe = [
    pattern for pattern in unsafe_patterns
    if pattern in all_code
]

assert not found_unsafe, (
    f"Review path-changing operations: {found_unsafe}"
)

assert str(PROJECT_ROOT) not in all_code, (
    "Original project path referenced in copied notebook."
)

print("Basic path safety checks: PASS")

# 3. Record original database fingerprints
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

original_databases = sorted(
    WAREHOUSE_DIR.glob("*.duckdb")
)

original_hashes = {
    str(path): sha256_file(path)
    for path in original_databases
}

print(
    "Original databases fingerprinted:",
    len(original_hashes)
)

# 4. Prepare a separate Python execution script
# Using sys.executable ensures the current .venv is used.

runner_code = r'''
import json
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

notebook_path = Path(
    "10_reproducible_data_pipeline.ipynb"
)

notebook = json.loads(
    notebook_path.read_text(encoding="utf-8")
)

namespace = {
    "__name__": "__main__",
    "display": display
}

for cell_number, cell in enumerate(
    notebook["cells"], start=1
):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if not source.strip():
        continue

    print(
        f"\n===== RUNNING CELL {cell_number} =====",
        flush=True
    )

    exec(
        compile(
            source,
            f"notebook_cell_{cell_number}",
            "exec"
        ),
        namespace,
        namespace
    )

print("\n===== ISOLATED NOTEBOOK EXECUTION COMPLETE =====")
'''

print("\n===== EXECUTING REBUILD =====")

try:
    result = subprocess.run(
        [sys.executable, "-c", runner_code],
        cwd=str(ISOLATED_ROOT),
        capture_output=True,
        text=True,
        timeout=900
    )

    execution_status = (
        "PASS" if result.returncode == 0
        else "REVIEW"
    )

    print("Execution status:", execution_status)
    print("Exit code:", result.returncode)

    print("\n===== REBUILD OUTPUT (FINAL SECTION) =====")
    print(result.stdout[-6000:])

    if result.stderr.strip():
        print("\n===== EXECUTION ERRORS =====")
        print(result.stderr[-3500:])

except subprocess.TimeoutExpired:
    execution_status = "REVIEW"
    print("REVIEW: Execution exceeded the safety timeout.")

# 5. Inspect generated warehouse artifacts
print("\n===== ISOLATED WAREHOUSE FILES =====")

isolated_databases = sorted(
    isolated_warehouse.glob("*.duckdb")
)

for path in isolated_databases:
    print(
        path.name,
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )

candidate_path = (
    isolated_warehouse
    / "demand_intelligence_rebuild_candidate.duckdb"
)

candidate_created = candidate_path.exists()

print(
    "\nRebuild candidate created:",
    "PASS" if candidate_created else "REVIEW"
)

# 6. Confirm original databases were not changed
original_unchanged = all(
    Path(path).exists()
    and sha256_file(Path(path)) == fingerprint
    for path, fingerprint in original_hashes.items()
)

print("\n===== ORIGINAL FILE INTEGRITY =====")
print(
    "Original database fingerprints unchanged:",
    "PASS" if original_unchanged else "REVIEW"
)

# 7. Final status
print("\n===== STEP 15 VALIDATION SUMMARY =====")

step15_pass = (
    execution_status == "PASS"
    and candidate_created
    and original_unchanged
)

if step15_pass:
    print("===== STEP 15 COMPLETE: PASS =====")
    print("Isolated rebuild executed successfully.")
    print("Original warehouse files unchanged.")
else:
    print("===== STEP 15 REQUIRES REVIEW =====")
    print("Inspect execution output before proceeding.")


===== ISOLATED WAREHOUSE REBUILD =====

===== EXECUTION ENVIRONMENT =====
Python: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/.venv/bin/python
Workspace: /private/var/folders/9c/495l1cs96zj_67bnbpkg5lm80000gn/T/hardware_demand_rebuild_4zsp3ig5
Notebook: 10_reproducible_data_pipeline.ipynb
Basic path safety checks: PASS
Original databases fingerprinted: 5

===== EXECUTING REBUILD =====
Execution status: PASS
Exit code: 0

===== REBUILD OUTPUT (FINAL SECTION) =====
 reporting.v_sensitivity_decision_cases
PASS: reporting.v_sensitivity_decision_coverage
PASS: reporting.v_sensitivity_economic_tradeoffs
PASS: reporting.v_sensitivity_intervention_evaluations

===== DATA PARITY VALIDATION SUMMARY =====
  Object_Type Status  Objects
0       TABLE   PASS       40
1        VIEW   PASS       16

===== OBJECTS REQUIRING REVIEW =====
None. All objects passed.

===== BUSINESS QUERY RECONCILIATION =====
                          Metric  Original  Rebuilt  Expected

In [21]:
# STEP 16: FRESH REBUILD VS FINAL WAREHOUSE

# READ ONLY — compare reporting views and their contents

import duckdb
import pandas as pd
import numpy as np
from datetime import date, datetime

print("===== FRESH REBUILD RECONCILIATION =====")

fresh_path = (
    ISOLATED_ROOT
    / "data"
    / "warehouse"
    / "demand_intelligence_rebuild_candidate.duckdb"
)

final_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert fresh_path.exists(), "Fresh rebuild database missing."
assert final_path.exists(), "Final warehouse missing."
assert fresh_path.resolve() != final_path.resolve()

# 1. Open both databases in read-only mode
fresh_con = duckdb.connect(
    str(fresh_path), read_only=True
)

final_con = duckdb.connect(
    str(final_path), read_only=True
)

try:
    view_sql = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'reporting'
          AND table_type = 'VIEW'
        ORDER BY table_name
    """

    fresh_views = set(
        fresh_con.execute(view_sql)
        .df()["table_name"]
    )

    final_views = set(
        final_con.execute(view_sql)
        .df()["table_name"]
    )

    shared_views = sorted(fresh_views & final_views)
    missing_views = sorted(final_views - fresh_views)
    extra_views = sorted(fresh_views - final_views)

    print("\n===== REPORTING VIEW INVENTORY =====")
    print("Fresh rebuild views:", len(fresh_views))
    print("Final warehouse views:", len(final_views))
    print("Shared views:", len(shared_views))

    print("\n===== VIEWS MISSING FROM FRESH REBUILD =====")
    print(missing_views if missing_views else "None")

    print("\n===== EXTRA VIEWS IN FRESH REBUILD =====")
    print(extra_views if extra_views else "None")

    # 2. Normalize values for order-independent comparisons
    def normalize_value(value):

        if value is None:
            return "<NULL>"

        if isinstance(value, (bool, np.bool_)):
            return f"BOOL:{bool(value)}"

        if isinstance(value, (int, float, np.number)):
            if pd.isna(value):
                return "<NULL>"
            return f"NUM:{float(value):.6f}"

        if isinstance(value, (date, datetime, pd.Timestamp)):
            return f"DATE:{pd.Timestamp(value).isoformat()}"

        if pd.isna(value):
            return "<NULL>"

        return f"TEXT:{str(value)}"

    def normalize_table(df):
        cols = sorted(df.columns)

        records = [
            tuple(
                normalize_value(value)
                for value in record
            )
            for record in df[cols].itertuples(
                index=False, name=None
            )
        ]

        return sorted(records)

    # 3. Compare every shared view
    comparison = []

    for view in shared_views:

        quoted = '"' + view.replace('"', '""') + '"'
        query = f'SELECT * FROM "reporting".{quoted}'

        try:
            fresh_df = fresh_con.execute(query).df()
            final_df = final_con.execute(query).df()

            columns_match = (
                set(fresh_df.columns)
                == set(final_df.columns)
            )

            rows_match = (
                len(fresh_df) == len(final_df)
            )

            content_match = False

            if columns_match and rows_match:
                content_match = (
                    normalize_table(fresh_df)
                    == normalize_table(final_df)
                )

            passed = (
                columns_match
                and rows_match
                and content_match
            )

            comparison.append({
                "View": view,
                "Fresh_Rows": len(fresh_df),
                "Final_Rows": len(final_df),
                "Columns_Match": columns_match,
                "Content_Match": content_match,
                "Status": "PASS" if passed else "REVIEW"
            })

        except Exception as error:
            comparison.append({
                "View": view,
                "Fresh_Rows": None,
                "Final_Rows": None,
                "Columns_Match": False,
                "Content_Match": False,
                "Status": "ERROR"
            })
            print(f"Error in {view}: {error}")

finally:
    fresh_con.close()
    final_con.close()

# 4. Display results
comparison_df = pd.DataFrame(comparison)

print("\n===== SHARED VIEW RECONCILIATION =====")
display(comparison_df)

shared_parity = (
    len(comparison_df) == len(shared_views)
    and len(shared_views) > 0
    and comparison_df["Status"].eq("PASS").all()
)

full_parity = (
    shared_parity
    and fresh_views == final_views
)

print("\n===== STEP 16 VALIDATION SUMMARY =====")

print(
    "Shared view content parity:",
    "PASS" if shared_parity else "REVIEW"
)

print(
    "Complete final reporting coverage:",
    "PASS" if full_parity else "INCOMPLETE"
)

print("Shared views checked:", len(shared_views))
print("Missing final views:", len(missing_views))

if shared_parity and not full_parity:
    print("\n===== STEP 16: PARTIAL PARITY CONFIRMED =====")
    print("Shared reporting results match.")
    print("Additional reporting views require reconciliation.")

elif full_parity:
    print("\n===== STEP 16 COMPLETE: FULL PASS =====")
    print("All final reporting views reproduced.")

else:
    print("\n===== STEP 16 REQUIRES REVIEW =====")
    print("Inspect the view comparison table.")

print("\nRead-only comparison complete.")
print("No warehouse files modified.")


===== FRESH REBUILD RECONCILIATION =====

===== REPORTING VIEW INVENTORY =====
Fresh rebuild views: 16
Final warehouse views: 18
Shared views: 16

===== VIEWS MISSING FROM FRESH REBUILD =====
['v_external_signal_aligned_demand', 'v_external_signal_model_comparison']

===== EXTRA VIEWS IN FRESH REBUILD =====
None

===== SHARED VIEW RECONCILIATION =====


,View,Fresh_Rows,Final_Rows,Columns_Match,Content_Match,Status
0,v_airpods_weekly_demand,51,51,True,True,PASS
1,v_forecast_lead_time_baseline,4,4,True,True,PASS
2,v_forecast_model_performance,8,8,True,True,PASS
3,v_forecast_multihorizon_performance,15,15,True,True,PASS
4,v_headphone_daily_signals,61,61,True,True,PASS
5,v_inventory_candidate_actions,2,2,True,True,PASS
6,v_inventory_launch_readiness,7,7,True,True,PASS
7,v_inventory_scenario_performance,3,3,True,True,PASS
8,v_inventory_weekly,12,12,True,True,PASS
9,v_purchase_intent_models,7,7,True,True,PASS



===== STEP 16 VALIDATION SUMMARY =====
Shared view content parity: PASS
Complete final reporting coverage: INCOMPLETE
Shared views checked: 16
Missing final views: 2

===== STEP 16: PARTIAL PARITY CONFIRMED =====
Shared reporting results match.
Additional reporting views require reconciliation.

Read-only comparison complete.
No warehouse files modified.


In [ ]:
# STEP 17: TRACE MISSING EXTERNAL-SIGNAL VIEWS

# Read-only inspection of notebooks and final DuckDB

import json
import duckdb
from pathlib import Path

print("===== EXTERNAL SIGNAL VIEW TRACE =====")

missing_views = [
    "v_external_signal_aligned_demand",
    "v_external_signal_model_comparison"
]

# 1. Search existing notebooks for the view definitions
print("\n===== NOTEBOOK REFERENCES =====")

notebooks = sorted(
    p for p in PROJECT_ROOT.glob("*.ipynb")
    if p.name != "13_final_project_validation.ipynb"
)

matches_found = 0

for notebook_path in notebooks:

    nb = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    for cell_num, cell in enumerate(
        nb.get("cells", []), start=1
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        hit_indices = [
            i for i, line in enumerate(lines)
            if any(
                name.lower() in line.lower()
                for name in missing_views
            )
        ]

        if not hit_indices:
            continue

        matches_found += 1

        print(
            f"\nNotebook: {notebook_path.name}"
            f" | Cell: {cell_num}"
        )

        selected = set()

        for i in hit_indices[:5]:
            for j in range(
                max(0, i - 5),
                min(len(lines), i + 9)
            ):
                selected.add(j)

        previous = -2

        for j in sorted(selected):
            if j > previous + 1:
                print("...")

            print(
                f"{j+1:>4}: {lines[j][:180]}"
            )

            previous = j

print("\nMatching notebook cells:", matches_found)

# 2. Inspect view definitions in the final warehouse
final_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

con = duckdb.connect(
    str(final_path),
    read_only=True
)

try:
    definitions = con.execute("""
        SELECT
            view_name,
            sql
        FROM duckdb_views()
        WHERE schema_name = 'reporting'
          AND view_name IN (
              'v_external_signal_aligned_demand',
              'v_external_signal_model_comparison'
          )
        ORDER BY view_name
    """).df()

finally:
    con.close()

print("\n===== FINAL VIEW SQL DEFINITIONS =====")

for _, row in definitions.iterrows():

    print(f"\nVIEW: {row['view_name']}")
    print(str(row["sql"])[:2500])

print("\nDefinitions found:", len(definitions))

# 3. Inspect the original external-signals notebook
print("\n===== NOTEBOOK 11 INVENTORY =====")

signal_notebooks = sorted(
    PROJECT_ROOT.glob("11_*.ipynb")
)

for path in signal_notebooks:

    nb = json.loads(
        path.read_text(encoding="utf-8")
    )

    code_cells = [
        "".join(cell.get("source", []))
        for cell in nb.get("cells", [])
        if cell.get("cell_type") == "code"
    ]

    print("Notebook:", path.name)
    print("Code cells:", len(code_cells))

    full_code = "\n".join(code_cells).lower()

    for label, term in {
        "External signal processing": "external_signal",
        "Reporting view creation": "create",
        "DuckDB connection": "duckdb.connect",
        "Processed CSV inputs": "read_csv",
        "SQL transformations": "select"
    }.items():
        print(
            f"{label}:",
            "FOUND" if term in full_code else "NOT DETECTED"
        )

print("\n===== STEP 17 COMPLETE =====")
print("Missing reporting-view origins inspected.")
print("No source files or databases modified.")


===== EXTERNAL SIGNAL VIEW TRACE =====

===== NOTEBOOK REFERENCES =====

Notebook: 11_external_demand_signals.ipynb | Cell: 6
...
 310:     # PART 8: CREATE EXTERNAL MODEL REPORTING VIEW
 311:     # --------------------------------------------------------
 312: 
 313:     con.execute("""
 314:         CREATE OR REPLACE VIEW
 315:         reporting.v_external_signal_model_comparison AS
 316: 
 317:         SELECT
 318:             Model,
 319:             Forecasts,
 320:             MAE,
 321:             WAPE_Pct,
 322:             Bias,
 323:             WAPE_Gap_vs_Naive_pp,
...
 340:     # PART 9: CREATE ALIGNED DEMAND REPORTING VIEW
 341:     # --------------------------------------------------------
 342: 
 343:     con.execute("""
 344:         CREATE OR REPLACE VIEW
 345:         reporting.v_external_signal_aligned_demand AS
 346: 
 347:         SELECT
 348:             Week_Ending,
 349:             Units_Sold,
 350:             Forecast_Origin,
 351:             Overlapping_W

In [28]:

# STEP 18: REPRODUCE FINAL EXTERNAL-SIGNAL REPORTING VIEWS
# Uses SQL from Notebook 11 against the isolated rebuild.
# Does not modify the original project databases.

import ast
import json
import re
import duckdb
import pandas as pd

print("===== FINAL EXTERNAL SIGNAL VIEW REBUILD =====")

# 1. Identify the two missing reporting views
target_views = [
    "v_external_signal_aligned_demand",
    "v_external_signal_model_comparison"
]

source_notebook = (
    PROJECT_ROOT / "11_external_demand_signals.ipynb"
)

fresh_path = (
    ISOLATED_ROOT
    / "data"
    / "warehouse"
    / "demand_intelligence_rebuild_candidate.duckdb"
)

final_path = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert source_notebook.exists()
assert fresh_path.exists()
assert final_path.exists()

# Safety: only permit writes inside the temporary workspace
assert fresh_path.resolve().is_relative_to(
    ISOLATED_ROOT.resolve()
)

assert fresh_path.resolve() != final_path.resolve()

print("\n===== DATABASE SAFETY =====")
print("Target:", fresh_path.name)
print("Isolated location: PASS")

# 2. Extract SQL statements from Notebook 11
nb = json.loads(
    source_notebook.read_text(encoding="utf-8")
)

view_sql = {}

for cell in nb["cells"]:

    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        syntax_tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in ast.walk(syntax_tree):

        if not isinstance(node, ast.Call):
            continue

        if not isinstance(node.func, ast.Attribute):
            continue

        if node.func.attr != "execute" or not node.args:
            continue

        argument = node.args[0]

        if not (
            isinstance(argument, ast.Constant)
            and isinstance(argument.value, str)
        ):
            continue

        sql = argument.value.strip()

        for name in target_views:

            pattern = (
                r"^\s*CREATE\s+OR\s+REPLACE\s+VIEW\s+"
                r'"?reporting"?\s*\.\s*"?'
                + re.escape(name)
                + r'"?\s+AS\b'
            )

            if re.search(pattern, sql, re.IGNORECASE):
                assert name not in view_sql, (
                    f"Duplicate SQL definition: {name}"
                )

                view_sql[name] = sql

print("\n===== ORIGINAL SQL DEFINITIONS =====")

for name in target_views:
    print(
        name,
        "FOUND" if name in view_sql else "MISSING"
    )

assert set(view_sql) == set(target_views), (
    "Original SQL definitions not fully located. "
    "Review Notebook 11 before proceeding."
)

# 3. Open isolated rebuild and final warehouse
fresh_con = duckdb.connect(
    str(fresh_path),
    read_only=False
)

final_con = duckdb.connect(
    str(final_path),
    read_only=True
)

try:
    # Start a transaction so failed validation
    # does not leave partially created views.
    fresh_con.execute("BEGIN TRANSACTION")

    # Confirm target views are not already present
    existing = {
        row[0]
        for row in fresh_con.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'reporting'
              AND table_type = 'VIEW'
        """).fetchall()
    }

    assert not (set(target_views) & existing), (
        "Target views already exist. "
        "Do not overwrite without reviewing."
    )

    # 4. Recreate views using original notebook SQL
    print("\n===== RECREATING REPORTING VIEWS =====")

    for name in target_views:
        fresh_con.execute(view_sql[name])
        print(name, "CREATED")

    # 5. Compare the newly created views
    print("\n===== EXTERNAL VIEW RECONCILIATION =====")

    results = []

    for name in target_views:

        quoted = '"' + name.replace('"', '""') + '"'

        query = (
            f'SELECT * FROM "reporting".{quoted}'
        )

        fresh_df = fresh_con.execute(query).df()
        final_df = final_con.execute(query).df()

        columns_match = (
            set(fresh_df.columns)
            == set(final_df.columns)
        )

        content_match = False

        if columns_match and len(fresh_df) == len(final_df):
            content_match = (
                normalize_table(fresh_df)
                == normalize_table(final_df)
            )

        results.append({
            "View": name,
            "Fresh_Rows": len(fresh_df),
            "Final_Rows": len(final_df),
            "Content_Match": content_match,
            "Status": (
                "PASS" if content_match else "REVIEW"
            )
        })

    result_df = pd.DataFrame(results)
    display(result_df)

    external_pass = result_df["Status"].eq(
        "PASS"
    ).all()

    # 6. Confirm all 18 reporting views are now present
    final_view_count = fresh_con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = 'reporting'
          AND table_type = 'VIEW'
    """).fetchone()[0]

    full_pass = (
        external_pass
        and final_view_count == 18
    )

    if full_pass:
        fresh_con.execute("COMMIT")
    else:
        fresh_con.execute("ROLLBACK")

except Exception:
    fresh_con.execute("ROLLBACK")
    raise

finally:
    fresh_con.close()
    final_con.close()

# 7. Final report
print("\n===== STEP 18 VALIDATION SUMMARY =====")

print("Original SQL definitions:", len(view_sql))
print("Final reporting view count:", final_view_count)

print(
    "External view data parity:",
    "PASS" if external_pass else "REVIEW"
)

print(
    "All 18 reporting views available:",
    "PASS" if full_pass else "REVIEW"
)

if full_pass:
    print("\n===== STEP 18 COMPLETE: PASS =====")
    print("All 18 reporting views now reproduced.")
    print("Original repository databases unchanged.")
else:
    print("\n===== STEP 18 REQUIRES REVIEW =====")
    print("Changes rolled back in isolated database.")


===== FINAL EXTERNAL SIGNAL VIEW REBUILD =====

===== DATABASE SAFETY =====
Target: demand_intelligence_rebuild_candidate.duckdb
Isolated location: PASS

===== ORIGINAL SQL DEFINITIONS =====
v_external_signal_aligned_demand FOUND
v_external_signal_model_comparison FOUND

===== RECREATING REPORTING VIEWS =====
v_external_signal_aligned_demand CREATED
v_external_signal_model_comparison CREATED

===== EXTERNAL VIEW RECONCILIATION =====


,View,Fresh_Rows,Final_Rows,Content_Match,Status
0,v_external_signal_aligned_demand,51,51,True,PASS
1,v_external_signal_model_comparison,3,3,True,PASS



===== STEP 18 VALIDATION SUMMARY =====
Original SQL definitions: 2
Final reporting view count: 18
External view data parity: PASS
All 18 reporting views available: PASS

===== STEP 18 COMPLETE: PASS =====
All 18 reporting views now reproduced.
Original repository databases unchanged.


In [24]:

# STEP 18A: DIAGNOSE MISSING SIGNAL TABLES

# READ ONLY — no database modifications

import duckdb
import json

print("===== EXTERNAL SIGNAL DEPENDENCY DIAGNOSTIC =====")

# 1. Inspect signals schema in both databases

for label, db_path in [
    ("Isolated Rebuild", fresh_path),
    ("Final Warehouse", final_path)
]:

    con = duckdb.connect(
        str(db_path),
        read_only=True
    )

    try:
        objects = con.execute("""
            SELECT
                table_schema,
                table_name,
                table_type
            FROM information_schema.tables
            WHERE table_schema = 'signals'
            ORDER BY table_name
        """).df()

    finally:
        con.close()

    print(f"\n===== {label.upper()} =====")
    print("Signal objects found:", len(objects))

    if not objects.empty:
        display(objects)
    else:
        print("No signals tables or views found.")

# 2. Display SQL dependencies of the missing views

print("\n===== MISSING VIEW DEPENDENCIES =====")

for name in target_views:

    print(f"\nVIEW: {name}")

    if name in view_sql:
        print(view_sql[name][:3000])
    else:
        print("SQL definition not found.")

# 3. Inspect Notebook 11 for signal table creation

print("\n===== ORIGINAL SIGNAL BUILD LOGIC =====")

nb_path = (
    PROJECT_ROOT
    / "11_external_demand_signals.ipynb"
)

nb = json.loads(
    nb_path.read_text(encoding="utf-8")
)

keywords = [
    "create schema",
    "signals.",
    "external_signal_aligned_demand_2019",
    "create or replace table",
    "create table",
    "read_csv",
    "signals_v2"
]

for cell_number, cell in enumerate(
    nb["cells"], start=1
):

    if cell["cell_type"] != "code":
        continue

    lines = "".join(
        cell.get("source", [])
    ).splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(
            keyword in line.lower()
            for keyword in keywords
        )
    ]

    if not matches:
        continue

    print(f"\n===== CELL {cell_number} =====")
    print("Relevant lines:", len(matches))

    # Show the first 18 relevant lines
    # with surrounding context
    shown = set()

    for i in matches[:18]:

        for j in range(
            max(0, i - 2),
            min(len(lines), i + 4)
        ):

            if j not in shown:
                print(f"{j+1:>4}: {lines[j][:180]}")
                shown.add(j)

        print("...")

print("\n===== STEP 18A COMPLETE =====")
print("Signal dependencies inspected.")
print("No database files modified.")


===== EXTERNAL SIGNAL DEPENDENCY DIAGNOSTIC =====

===== ISOLATED REBUILD =====
Signal objects found: 0
No signals tables or views found.

===== FINAL WAREHOUSE =====
Signal objects found: 6


,table_schema,table_name,table_type
0,signals,external_signal_aligned_demand_2019,BASE TABLE
1,signals,external_signal_backtest_predictions_2019,BASE TABLE
2,signals,external_signal_model_comparison_2019,BASE TABLE
3,signals,external_signal_stability_2019,BASE TABLE
4,signals,external_signal_weekly_diagnostics_2019,BASE TABLE
5,signals,google_trends_airpods_weekly_2019,BASE TABLE



===== MISSING VIEW DEPENDENCIES =====

VIEW: v_external_signal_aligned_demand
CREATE OR REPLACE VIEW
        reporting.v_external_signal_aligned_demand AS

        SELECT
            Week_Ending,
            Units_Sold,
            Forecast_Origin,
            Overlapping_Week_Search_Interest,
            Prior_Completed_Search_Interest,
            Boundary_Week_Flag

        FROM signals.external_signal_aligned_demand_2019

VIEW: v_external_signal_model_comparison
CREATE OR REPLACE VIEW
        reporting.v_external_signal_model_comparison AS

        SELECT
            Model,
            Forecasts,
            MAE,
            WAPE_Pct,
            Bias,
            WAPE_Gap_vs_Naive_pp,

            CASE
                WHEN Model = 'Naive Forecast'
                    THEN 'Benchmark'

                WHEN WAPE_Gap_vs_Naive_pp < 0
                    THEN 'Better than naive'

                ELSE 'Worse than or equal to naive'
            END AS Benchmark_Interpretation

        F

In [ ]:

# STEP 18B: VERIFY EXTERNAL SIGNAL SOURCE FILES

# Read-only preflight before rebuilding missing tables

import pandas as pd
import duckdb

print("===== EXTERNAL SIGNAL SOURCE VALIDATION =====")

required_signal_tables = [
    "external_signal_aligned_demand_2019",
    "external_signal_backtest_predictions_2019",
    "external_signal_model_comparison_2019",
    "external_signal_stability_2019",
    "external_signal_weekly_diagnostics_2019",
    "google_trends_airpods_weekly_2019"
]

# 1. Inspect required source CSV availability
source_inventory = []

for table_name in required_signal_tables:

    csv_path = PROCESSED_DIR / f"{table_name}.csv"

    source_inventory.append({
        "Table": table_name,
        "CSV_Exists": csv_path.exists(),
        "CSV_Path": str(csv_path),
        "CSV_Rows": (
            sum(1 for _ in open(csv_path, encoding="utf-8-sig")) - 1
            if csv_path.exists()
            else None
        )
    })

inventory_df = pd.DataFrame(source_inventory)

print("\n===== PROCESSED CSV INVENTORY =====")
display(
    inventory_df[
        ["Table", "CSV_Exists", "CSV_Rows"]
    ]
)

# 2. Compare source CSV fields against final table fields
con = duckdb.connect(
    str(final_path),
    read_only=True
)

schema_results = []

try:
    for table_name in required_signal_tables:

        csv_path = PROCESSED_DIR / f"{table_name}.csv"

        final_columns = con.execute("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'signals'
              AND table_name = ?
            ORDER BY ordinal_position
        """, [table_name]).df()["column_name"].tolist()

        final_count = con.execute(
            f'SELECT COUNT(*) FROM "signals"."{table_name}"'
        ).fetchone()[0]

        if csv_path.exists():
            csv_columns = pd.read_csv(
                csv_path,
                nrows=0
            ).columns.tolist()

            csv_count = len(pd.read_csv(csv_path))

            columns_match = (
                csv_columns == final_columns
            )

            counts_match = (
                csv_count == final_count
            )

            status = (
                "PASS"
                if columns_match and counts_match
                else "REVIEW"
            )
        else:
            csv_columns = []
            csv_count = None
            columns_match = False
            counts_match = False
            status = "MISSING"

        schema_results.append({
            "Table": table_name,
            "CSV_Rows": csv_count,
            "Final_Rows": final_count,
            "Columns_Match": columns_match,
            "Rows_Match": counts_match,
            "Status": status
        })

finally:
    con.close()

print("\n===== CSV VS FINAL TABLE CHECK =====")

schema_df = pd.DataFrame(schema_results)
display(schema_df)

# 3. Find any missing CSVs
missing = inventory_df.loc[
    ~inventory_df["CSV_Exists"],
    "Table"
].tolist()

if missing:
    print("\n===== MISSING SOURCE FILES =====")
    for name in missing:
        print(name)

    print("\nPossible related processed files:")

    for path in sorted(PROCESSED_DIR.glob("*.csv")):
        if any(
            word in path.name.lower()
            for word in ["external", "signal", "trends"]
        ):
            print(path.name)

# 4. Final preflight result
preflight_pass = schema_df["Status"].eq("PASS").all()

print("\n===== STEP 18B VALIDATION =====")

print(
    "Six signal sources ready:",
    "PASS" if preflight_pass else "REVIEW"
)

if preflight_pass:
    print("Ready to reconstruct isolated signal tables.")
else:
    print("Inspect missing files or schema differences.")

print("No database files modified.")


===== EXTERNAL SIGNAL SOURCE VALIDATION =====

===== PROCESSED CSV INVENTORY =====


,Table,CSV_Exists,CSV_Rows
0,external_signal_aligned_demand_2019,True,51
1,external_signal_backtest_predictions_2019,True,19
2,external_signal_model_comparison_2019,True,3
3,external_signal_stability_2019,True,6
4,external_signal_weekly_diagnostics_2019,True,19
5,google_trends_airpods_weekly_2019,True,53



===== CSV VS FINAL TABLE CHECK =====


,Table,CSV_Rows,Final_Rows,Columns_Match,Rows_Match,Status
0,external_signal_aligned_demand_2019,51,51,True,True,PASS
1,external_signal_backtest_predictions_2019,19,19,True,True,PASS
2,external_signal_model_comparison_2019,3,3,True,True,PASS
3,external_signal_stability_2019,6,6,True,True,PASS
4,external_signal_weekly_diagnostics_2019,19,19,True,True,PASS
5,google_trends_airpods_weekly_2019,53,53,True,True,PASS



===== STEP 18B VALIDATION =====
Six signal sources ready: PASS
Ready to reconstruct isolated signal tables.
No database files modified.


In [27]:

# STEP 18C: REBUILD MISSING SIGNAL TABLES
# Writes ONLY to the isolated DuckDB database
# Imports source CSVs and verifies full table contents

import duckdb
import pandas as pd
from pathlib import Path

print("===== ISOLATED SIGNAL TABLE REBUILD =====")

# 1. Confirm database paths
fresh_db = (
    ISOLATED_ROOT
    / "data"
    / "warehouse"
    / "demand_intelligence_rebuild_candidate.duckdb"
).resolve()

final_db = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
).resolve()

isolated_csv_dir = (
    ISOLATED_ROOT / "data" / "processed"
).resolve()

assert fresh_db.exists()
assert final_db.exists()
assert isolated_csv_dir.exists()

assert fresh_db != final_db
assert fresh_db.is_relative_to(ISOLATED_ROOT.resolve())
assert isolated_csv_dir.is_relative_to(
    ISOLATED_ROOT.resolve()
)

# All source files must be inside the isolated workspace
signal_tables = [
    "external_signal_aligned_demand_2019",
    "external_signal_backtest_predictions_2019",
    "external_signal_model_comparison_2019",
    "external_signal_stability_2019",
    "external_signal_weekly_diagnostics_2019",
    "google_trends_airpods_weekly_2019"
]

for table in signal_tables:
    assert (isolated_csv_dir / f"{table}.csv").exists()

print("\n===== SAFETY VALIDATION =====")
print("Isolated database location: PASS")
print("All six copied CSV inputs available: PASS")
print("Final database is reference-only: PASS")

# Safely quote SQL file paths
def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"

# 2. Open isolated database for writing
con = duckdb.connect(str(fresh_db), read_only=False)

results = []
rebuild_pass = False
transaction_open = False

try:
    # Final database is attached READ ONLY for comparison
    con.execute(
        f"ATTACH {sql_literal(final_db)} "
        "AS validation_ref (READ_ONLY)"
    )

    con.execute("BEGIN TRANSACTION")
    transaction_open = True

    # 3. Create missing signals schema
    con.execute(
        "CREATE SCHEMA IF NOT EXISTS signals"
    )

    print("\n===== REBUILDING SIGNAL TABLES =====")

    for table in signal_tables:
        quoted = '"' + table.replace('"', '""') + '"'

        csv_path = isolated_csv_dir / f"{table}.csv"

        # Prevent overwriting an existing isolated table
        existing = con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_catalog = current_database()
        AND table_schema = 'signals'
        AND table_name = ?
    """, [table]).fetchone()[0]

        assert existing == 0, (
            f"Table already exists: signals.{table}"
        )

        # Reconstruct from the COPIED processed CSV
        con.execute(f"""
            CREATE TABLE signals.{quoted} AS
            SELECT *
            FROM read_csv_auto(
                {sql_literal(csv_path)},
                header = true,
                sample_size = -1
            )
        """)

        # 4. Compare rebuilt table with final reference
        fresh_query = f"signals.{quoted}"
        reference_query = (
            f"validation_ref.signals.{quoted}"
        )

        fresh_count = con.execute(
            f"SELECT COUNT(*) FROM {fresh_query}"
        ).fetchone()[0]

        reference_count = con.execute(
            f"SELECT COUNT(*) FROM {reference_query}"
        ).fetchone()[0]

        fresh_cols = con.execute(
            f"SELECT * FROM {fresh_query} LIMIT 0"
        ).df().columns.tolist()

        reference_cols = con.execute(
            f"SELECT * FROM {reference_query} LIMIT 0"
        ).df().columns.tolist()

        columns_match = fresh_cols == reference_cols
        rows_match = fresh_count == reference_count

        content_match = False

        if columns_match:
            # EXCEPT ALL checks every row, including duplicates.
            # Test both directions for exact data parity.
            fresh_only = con.execute(f"""
                SELECT COUNT(*)
                FROM (
                    SELECT * FROM {fresh_query}
                    EXCEPT ALL
                    SELECT * FROM {reference_query}
                )
            """).fetchone()[0]

            reference_only = con.execute(f"""
                SELECT COUNT(*)
                FROM (
                    SELECT * FROM {reference_query}
                    EXCEPT ALL
                    SELECT * FROM {fresh_query}
                )
            """).fetchone()[0]

            content_match = (
                fresh_only == 0
                and reference_only == 0
            )

        passed = (
            columns_match
            and rows_match
            and content_match
        )

        results.append({
            "Table": table,
            "Fresh_Rows": fresh_count,
            "Reference_Rows": reference_count,
            "Columns_Match": columns_match,
            "Content_Match": content_match,
            "Status": "PASS" if passed else "REVIEW"
        })

        print(
            f"{table}:",
            "PASS" if passed else "REVIEW"
        )

    # 5. Commit only if all six tables match
    result_df = pd.DataFrame(results)

    rebuild_pass = (
        len(result_df) == 6
        and result_df["Status"].eq("PASS").all()
    )

    if rebuild_pass:
        con.execute("COMMIT")
        print("\nAll six tables committed successfully.")
    else:
        con.execute("ROLLBACK")
        print("\nValidation failed. Import rolled back.")

    transaction_open = False

except Exception:
    if transaction_open:
        con.execute("ROLLBACK")
    raise

finally:
    con.close()

# 6. Final audit summary
print("\n===== TABLE RECONCILIATION =====")
display(result_df)

print("\n===== STEP 18C VALIDATION =====")

print(
    "All six signal tables reconstructed:",
    "PASS" if rebuild_pass else "REVIEW"
)

print(
    "Complete source-to-reference data parity:",
    "PASS" if rebuild_pass else "REVIEW"
)

if rebuild_pass:
    print("\n===== STEP 18C COMPLETE: PASS =====")
    print("Six signal tables loaded into isolated DuckDB.")
    print("Original repository databases unchanged.")
    print("Ready to recreate the final reporting views.")
else:
    print("\n===== STEP 18C REQUIRES REVIEW =====")
    print("No partial signal-table import retained.")


===== ISOLATED SIGNAL TABLE REBUILD =====

===== SAFETY VALIDATION =====
Isolated database location: PASS
All six copied CSV inputs available: PASS
Final database is reference-only: PASS

===== REBUILDING SIGNAL TABLES =====
external_signal_aligned_demand_2019: PASS
external_signal_backtest_predictions_2019: PASS
external_signal_model_comparison_2019: PASS
external_signal_stability_2019: PASS
external_signal_weekly_diagnostics_2019: PASS
google_trends_airpods_weekly_2019: PASS

All six tables committed successfully.

===== TABLE RECONCILIATION =====


,Table,Fresh_Rows,Reference_Rows,Columns_Match,Content_Match,Status
0,external_signal_aligned_demand_2019,51,51,True,True,PASS
1,external_signal_backtest_predictions_2019,19,19,True,True,PASS
2,external_signal_model_comparison_2019,3,3,True,True,PASS
3,external_signal_stability_2019,6,6,True,True,PASS
4,external_signal_weekly_diagnostics_2019,19,19,True,True,PASS
5,google_trends_airpods_weekly_2019,53,53,True,True,PASS



===== STEP 18C VALIDATION =====
All six signal tables reconstructed: PASS
Complete source-to-reference data parity: PASS

===== STEP 18C COMPLETE: PASS =====
Six signal tables loaded into isolated DuckDB.
Original repository databases unchanged.
Ready to recreate the final reporting views.


In [29]:
# STEP 19: FINAL END-TO-END REPORTING RECONCILIATION

# READ ONLY — verifies all 18 reporting views

import duckdb
import pandas as pd

print("===== FINAL REPORTING RECONCILIATION =====")

fresh_db = (
    ISOLATED_ROOT
    / "data"
    / "warehouse"
    / "demand_intelligence_rebuild_candidate.duckdb"
)

final_db = (
    WAREHOUSE_DIR
    / "demand_intelligence_signals_manifest_rebuild.duckdb"
)

assert fresh_db.exists()
assert final_db.exists()
assert fresh_db.resolve() != final_db.resolve()

# 1. Connect to both databases in READ-ONLY mode
fresh_con = duckdb.connect(
    str(fresh_db), read_only=True
)

final_con = duckdb.connect(
    str(final_db), read_only=True
)

try:
    view_query = """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'reporting'
          AND table_type = 'VIEW'
        ORDER BY table_name
    """

    fresh_views = set(
        fresh_con.execute(view_query)
        .df()["table_name"]
    )

    final_views = set(
        final_con.execute(view_query)
        .df()["table_name"]
    )

    print("\n===== VIEW INVENTORY =====")
    print("Rebuilt reporting views:", len(fresh_views))
    print("Final reporting views:", len(final_views))

    inventory_pass = (
        fresh_views == final_views
        and len(fresh_views) == 18
    )

    print(
        "Reporting view inventory:",
        "PASS" if inventory_pass else "REVIEW"
    )

    # 2. Compare all view contents independently
    results = []

    for view in sorted(fresh_views & final_views):

        quoted = '"' + view.replace('"', '""') + '"'
        query = f'SELECT * FROM reporting.{quoted}'

        fresh_df = fresh_con.execute(query).df()
        final_df = final_con.execute(query).df()

        columns_match = (
            set(fresh_df.columns)
            == set(final_df.columns)
        )

        rows_match = (
            len(fresh_df) == len(final_df)
        )

        # Reuse the row normalization from Step 16.
        # Includes duplicate rows and is independent
        # of the query's output ordering.
        content_match = (
            columns_match
            and rows_match
            and normalize_table(fresh_df)
            == normalize_table(final_df)
        )

        passed = (
            columns_match
            and rows_match
            and content_match
        )

        results.append({
            "Reporting_View": view,
            "Rebuilt_Rows": len(fresh_df),
            "Final_Rows": len(final_df),
            "Content_Match": content_match,
            "Status": "PASS" if passed else "REVIEW"
        })

finally:
    fresh_con.close()
    final_con.close()

# 3. Display final results
final_audit = pd.DataFrame(results)

print("\n===== ALL REPORTING VIEW RESULTS =====")
display(final_audit)

matched_views = int(
    final_audit["Status"].eq("PASS").sum()
)

print("\n===== FINAL VALIDATION SUMMARY =====")

print("Views compared:", len(final_audit))
print("Views matching:", matched_views)

full_parity = (
    inventory_pass
    and matched_views == 18
)

print(
    "Complete reporting parity:",
    "PASS" if full_parity else "REVIEW"
)

# 4. Confirm original warehouse fingerprints
# against the baseline captured in Step 15
original_unchanged = all(
    sha256_file(Path(path)) == fingerprint
    for path, fingerprint in original_hashes.items()
)

print(
    "Original warehouse files unchanged:",
    "PASS" if original_unchanged else "REVIEW"
)

# 5. Final technical status
if full_parity and original_unchanged:
    print("\n===== STEP 19 COMPLETE: PASS =====")
    print("All 18 reporting views independently reconciled.")
    print("Rebuilt reporting warehouse matches final warehouse.")
    print("Original databases remain unchanged.")
else:
    print("\n===== STEP 19 REQUIRES REVIEW =====")

print("\nValidation scope: processed-input reporting rebuild.")


===== FINAL REPORTING RECONCILIATION =====

===== VIEW INVENTORY =====
Rebuilt reporting views: 18
Final reporting views: 18
Reporting view inventory: PASS

===== ALL REPORTING VIEW RESULTS =====


,Reporting_View,Rebuilt_Rows,Final_Rows,Content_Match,Status
0,v_airpods_weekly_demand,51,51,True,PASS
1,v_external_signal_aligned_demand,51,51,True,PASS
2,v_external_signal_model_comparison,3,3,True,PASS
3,v_forecast_lead_time_baseline,4,4,True,PASS
4,v_forecast_model_performance,8,8,True,PASS
5,v_forecast_multihorizon_performance,15,15,True,PASS
6,v_headphone_daily_signals,61,61,True,PASS
7,v_inventory_candidate_actions,2,2,True,PASS
8,v_inventory_launch_readiness,7,7,True,PASS
9,v_inventory_scenario_performance,3,3,True,PASS



===== FINAL VALIDATION SUMMARY =====
Views compared: 18
Views matching: 18
Complete reporting parity: PASS
Original warehouse files unchanged: PASS

===== STEP 19 COMPLETE: PASS =====
All 18 reporting views independently reconciled.
Rebuilt reporting warehouse matches final warehouse.
Original databases remain unchanged.

Validation scope: processed-input reporting rebuild.



# Final Technical Validation Summary

## Project: Hardware Demand-Sensing & Launch Readiness Planner

### 1. Validation Objective

This audit independently verified the consistency, integrity, and reporting reproducibility of a hardware demand-planning decision-support system.

The validation covered historical demand, forecasting performance, external demand signals, inventory simulation, replenishment economics, sensitivity analysis, and DuckDB reporting infrastructure.

### 2. Historical Demand & Forecasting

- Reconciled 51 weeks of historical AirPods demand.
- Verified 15,364 total historical units.
- Independently reproduced forecast accuracy metrics for five forecasting models.
- Confirmed the Naive Forecast as the best-performing original model, with 8.60% WAPE.
- Validated external demand-signal model reporting against the warehouse.
- Confirmed that Google Trends improved the Ridge model relative to sales-history-only Ridge, but did not outperform the naive benchmark.

### 3. Inventory & Replenishment Planning

- Independently validated 12 weekly inventory simulation records across three operating scenarios.
- Reproduced baseline, demand-surge, and supplier-delay fill rates and shortage totals.
- Reconciled seven replenishment decision options.
- Independently reproduced modeled economic exposure reductions relative to no intervention.
- Confirmed a maximum modeled exposure reduction of $47,000 for the supplier-delay expedite option.
- Verified the two executive candidate actions against the modeled launch-readiness criteria.

### 4. Sensitivity & Decision Robustness

- Validated 162 simulated operating cases.
- Reconciled 567 intervention evaluations.
- Confirmed 261 READY classifications using the original shortage-aware readiness logic.
- Reproduced action-level robustness and scenario-level decision coverage.
- Identified that a fill rate above 98% does not independently guarantee READY classification when shortage weeks remain.

### 5. Warehouse Integrity & Reproducibility

- Validated all 18 DuckDB reporting views.
- Successfully executed the documented warehouse rebuild in an isolated workspace.
- Reconstructed six external-demand-signal tables from processed CSV inputs.
- Recreated two external-signal reporting views using the original SQL transformations.
- Independently reconciled all 18 rebuilt reporting views with the final reporting warehouse.
- Confirmed that the original warehouse databases remained unchanged during validation.

### 6. Scope & Limitations

- Historical and external-signal datasets do not necessarily describe the same product population.
- Forecasting performance is based on historical evaluation periods and does not guarantee future accuracy.
- Inventory scenarios and economic penalties rely on hypothetical planning assumptions.
- Modeled exposure reductions are not realized financial savings.
- Rebuild validation used processed inputs and a copied source database; complete raw-source-to-dashboard reproducibility was not established.
- Reporting comparisons used defined numerical comparison precision and should not be described as bit-for-bit database identity.

### 7. Final Conclusion

The project's historical demand metrics, forecasting evaluations, inventory simulations, replenishment decisions, sensitivity analyses, and reporting outputs successfully passed the completed consistency and reconciliation checks.

The isolated rebuild reproduced all 18 final DuckDB reporting views without altering the original databases.

These results support the technical reliability of the project's validated analytical outputs, within the stated data, modeling, and reproducibility limitations.

**Final status: Technical validation completed.**
